## 1~2일차 — 데이터 수집 및 전처리

### 데이터 출처
- 서울시 공공와이파이 서비스 위치 정보 (서울 열린데이터광장)
- 서울특별시 공공와이파이 AP별 사용량_고정형 (2021.05.01~10.13)
- 행정동 단위 서울 생활인구(내국인) 2021년 5~10월, 2026년 5월
- 행정동코드 매핑정보 (통계청코드 ↔ 행자부코드 연결)

### 전처리 핵심 사항
- 버스(시내버스·마을버스) 이동형 와이파이는 고정형 파일에 포함되지 않아 자동 제외
- 좌표 결측 AP 제거 후 9,226개 사용
- sjoin(within)으로 행정동 매칭 → 매칭 실패 48개를 sjoin_nearest로 보완
- 답십리1동·장지동 행정동코드 불일치를 직접 수정 (1106085→1106091, 1124070→1124081)
- 최종 기존지점: **395개 행정동**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'

loc = pd.read_csv(base_path + '서울시 공공와이파이 서비스 위치 정보.csv', encoding='cp949')
usage = pd.read_csv(base_path + '서울특별시 공공와이파이 AP별 사용량_고정형(20210501_20211013).csv', encoding='cp949')

print(loc.shape, usage.shape)

(26278, 13) (10191, 3)


In [ ]:
usage.head()

,관리번호,자치구,AP별 이용량(GB)
0,WF181037,성동구,13990.425250
1,WF181190,은평구,12011.047388
2,WF191009,은평구,10940.317724
3,WF181038,성동구,10244.118496
4,WF181035,성동구,9946.630856


In [ ]:
merged = pd.merge(usage, loc, on='관리번호', how='left')
print(merged.shape)
merged.head()

(10191, 15)


,관리번호,자치구_x,AP별 이용량(GB),자치구_y,와이파이명,도로명주소,설치유형,설치기관,서비스구분,망종류,설치년도,실내외구분,Y좌표,X좌표,작업일자
0,WF181037,성동구,13990.425250,성동구,서울특별시립비전트레이닝센터,자동차시장길48,6-1. 복지 - 사회,서울시(AP),공공WiFi,임대망,2018.0,실내,37.558437,127.055916,2026-06-20 15:12:05.0
1,WF181190,은평구,12011.047388,은평구,시립은평의마을,갈현로15길27-1,6-1. 복지 - 사회,서울시(AP),공공WiFi,임대망,2018.0,실내,37.607970,126.906425,2026-06-20 15:12:05.0
2,WF191009,은평구,10940.317724,은평구,시립은평의마을,갈현로15길 27-1(2생활관),6-1. 복지 - 사회,서울시(AP),공공WiFi,임대망,2019.0,실내,37.608433,126.906930,2026-06-20 15:12:05.0
3,WF181038,성동구,10244.118496,성동구,서울특별시립비전트레이닝센터,자동차시장길48,6-1. 복지 - 사회,서울시(AP),공공WiFi,임대망,2018.0,실내,37.558437,127.055916,2026-06-20 15:12:05.0
4,WF181035,성동구,9946.630856,성동구,서울특별시립비전트레이닝센터,자동차시장길48,6-1. 복지 - 사회,서울시(AP),공공WiFi,임대망,2018.0,실내,37.558437,127.055916,2026-06-20 15:12:05.0


In [ ]:
merged = merged.dropna(subset=['Y좌표', 'X좌표'])
print(merged.shape)

(9226, 15)


In [ ]:
print(merged['설치유형'].unique())

['6-1. 복지 - 사회' '6-4. 복지 - 아동청소년' '7-1-3. 공공 - 시산하기관' '7-1-1. 공공 - 시청'
 '6-3. 복지 - 장애인' '7-2-1. 공공 - 구청사 및 별관' '1. 주요거리' '6-5. 복지 - 여성다문화'
 '3. 공원(하천)' '4. 문화관광' '7-3. 공공 - 지역' '7-2-3. 공공 - 동주민센터' '2. 전통시장'
 '6-2. 복지 - 노인' '7-4. 공공 - 경제/커뮤니티' '7-2-2. 공공 - 구의회 및 보건소' '6-6. 복지 - 기타'
 '3. 공원(한강)' '5-2. 버스정류소(시비)' '7-2-4. 공공 - 구산하기관' '7-1-2. 공공 - 시의회'
 '5. 버스정류소']


In [ ]:
import geopandas as gpd

url = "https://raw.githubusercontent.com/southkorea/seoul-maps/master/kostat/2013/json/seoul_submunicipalities_geo_simple.json"
dong_boundary = gpd.read_file(url)
print(dong_boundary.columns.tolist())
dong_boundary.head()

['code', 'name', 'name_eng', 'base_year', 'geometry']


,code,name,name_eng,base_year,geometry
0,1125074,길동,Gil-dong,2013,"POLYGON ((127.15068 37.54298, 127.15666 37.539..."
1,1125073,천호2동,Cheonho 2(i)-dong,2013,"POLYGON ((127.12966 37.54732, 127.13669 37.544..."
2,1125072,암사1동,Amsa 1(il)-dong,2013,"POLYGON ((127.14659 37.55197, 127.14331 37.548..."
3,1125071,둔촌2동,Dunchon 2(i)-dong,2013,"POLYGON ((127.16252 37.5388, 127.15567 37.5312..."
4,1125070,둔촌1동,Dunchon 1(il)-dong,2013,"POLYGON ((127.13875 37.52602, 127.14684 37.523..."


In [ ]:
from shapely.geometry import Point

# merged 데이터를 geopandas 형태로 변환
# 주의: X좌표가 경도(longitude), Y좌표가 위도(latitude)입니다
merged['geometry'] = merged.apply(lambda row: Point(row['X좌표'], row['Y좌표']), axis=1)
wifi_gdf = gpd.GeoDataFrame(merged, geometry='geometry', crs='EPSG:4326')

# 좌표계를 행정동 경계 데이터와 통일
wifi_gdf = wifi_gdf.to_crs(dong_boundary.crs)

# 공간 조인: 각 와이파이 지점이 어느 행정동 폴리곤 안에 있는지 찾기
result = gpd.sjoin(wifi_gdf, dong_boundary, how='left', predicate='within')

print(result.shape)
result[['관리번호', 'name', 'code']].head(10)

(9226, 21)


,관리번호,name,code
0,WF181037,용답동,1104070
1,WF181190,구산동,1112057
2,WF191009,구산동,1112057
3,WF181038,용답동,1104070
4,WF181035,용답동,1104070
5,WF191010,구산동,1112057
6,WF190048,신월3동,1115059
8,WF190785,성북동,1108081
9,WF181034,용답동,1104070
10,WF181186,구산동,1112057


### sjoin_nearest 보완
sjoin(within)은 좌표가 폴리곤 경계 위에 찍힌 경우 매칭에 실패함.
매칭 실패한 48개 지점을 "가장 가까운 행정동"으로 재배정(sjoin_nearest)하여 전체 매칭 완료.

In [ ]:
실패 = wifi_gdf[~wifi_gdf['관리번호'].isin(result.dropna(subset=['name'])['관리번호'])]
재매칭 = gpd.sjoin_nearest(실패, dong_boundary, how='left')

성공 = result.dropna(subset=['name'])
result = pd.concat([성공, 재매칭], ignore_index=True)
print(result.shape)
print(result['name'].isna().sum())

(9227, 21)
0


/usr/local/lib/python3.12/dist-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [ ]:
print(result['name'].isna().sum())  # 매칭 안 된 행 개수
print(result['name'].nunique())  # 매칭된 행정동 개수

0
394


In [ ]:
result = result.dropna(subset=['name'])
print(result.shape)

(9227, 21)


In [ ]:
# 설치유형 대분류 매핑
유형_매핑 = {
    '1. 주요거리': '거리', '2. 전통시장': '상업',
    '3. 공원(하천)': '공원', '3. 공원(한강)': '공원',
    '4. 문화관광': '문화관광', '5. 버스정류소': '교통',
    '5-1. 버스정류소(국비)': '교통', '5-2. 버스정류소(시비)': '교통',
    '6-1. 복지 - 사회': '복지', '6-2. 복지 - 노인': '복지',
    '6-3. 복지 - 장애인': '복지', '6-4. 복지 - 아동청소년': '복지',
    '6-5. 복지 - 여성다문화': '복지', '6-6. 복지 - 기타': '복지',
    '7-1-1. 공공 - 시청': '공공', '7-1-2. 공공 - 시의회': '공공',
    '7-1-3. 공공 - 시산하기관': '공공', '7-2-1. 공공 - 구청사 및 별관': '공공',
    '7-2-2. 공공 - 구의회 및 보건소': '공공', '7-2-3. 공공 - 동주민센터': '공공',
    '7-2-3. 공공 - 주민센터': '공공', '7-2-4. 공공 - 구산하기관': '공공',
    '7-3. 공공 - 지역': '공공', '7-4. 공공 - 경제/커뮤니티': '공공',
}

# result에 설치유형 대분류 붙이기
wifi_location = pd.read_csv(base_path + '서울시 공공와이파이 서비스 위치 정보.csv', encoding='cp949')
wifi_location['설치유형_대분류'] = wifi_location['설치유형'].map(유형_매핑)
result = result.merge(wifi_location[['관리번호', '설치유형_대분류']], on='관리번호', how='left')

# 총이용량, AP개수 집계
dong_summary = result.groupby(['code', 'name']).agg(
    총이용량_GB=('AP별 이용량(GB)', 'sum'),
    AP개수=('관리번호', 'count')
).reset_index()

# 유형별 AP 개수 피벗
유형별_AP = result.groupby(['code', '설치유형_대분류']).size().unstack(fill_value=0).reset_index()
유형별_AP.columns.name = None

# 합치기
dong_summary['code'] = dong_summary['code'].astype(int)
유형별_AP['code'] = 유형별_AP['code'].astype(int)
dong_summary = dong_summary.merge(유형별_AP, on='code', how='left')

print(dong_summary.shape)
print(dong_summary.columns.tolist())
dong_summary.head(10)

(395, 11)
['code', 'name', '총이용량_GB', 'AP개수', '거리', '공공', '공원', '교통', '문화관광', '복지', '상업']


,code,name,총이용량_GB,AP개수,거리,공공,공원,교통,문화관광,복지,상업
0,1101053,사직동,93269.978605,246,16,190,2,0,38,0,0
1,1101054,삼청동,57.089743,1,0,0,0,0,1,0,0
2,1101055,부암동,29085.180390,53,2,48,3,0,0,0,0
3,1101056,평창동,1087.222709,18,12,0,3,0,0,3,0
4,1101057,무악동,3737.344281,10,0,0,0,0,10,0,0
5,1101058,교남동,119.451532,1,0,0,0,0,1,0,0
6,1101060,가회동,1041.933508,9,0,2,0,0,7,0,0
7,1101061,종로1·2·3·4가동,7838.411741,36,3,3,3,0,11,14,2
8,1101063,종로5·6가동,1462.391924,18,2,10,0,0,3,3,0
9,1101064,이화동,1152.575947,10,2,1,2,0,0,5,0


In [ ]:
dong_summary.to_csv('wifi_dong_summary.csv', index=False, encoding='utf-8-sig')

from google.colab import files
files.download('wifi_dong_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'

mapping = pd.read_excel(base_path + '행정동코드_매핑정보_20241218.xlsx',
                         sheet_name='행정동코드', skiprows=1)
mapping.columns = ['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명']
mapping.head()

,통계청코드,행자부코드,시도명,시군구명,행정동명
0,1101053,11110530,서울,종로구,사직동
1,1101054,11110540,서울,종로구,삼청동
2,1101055,11110550,서울,종로구,부암동
3,1101056,11110560,서울,종로구,평창동
4,1101057,11110570,서울,종로구,무악동


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/진로탐색프로젝트/민서'))

['와이파이', '서울시 공공와이파이 서비스 위치 정보.csv', '서울특별시 공공와이파이 AP별 사용량_고정형(20210501_20211013).csv', '이용량위치.csv', 'LOCAL_PEOPLE_DONG_202605.csv', '행정동코드_매핑정보_20241218.xlsx', '와이파이_생활인구_통합.csv', '생활인구 ', '신규후보지_2026.csv']


In [ ]:
base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/생활인구 /'

files_list = ['LOCAL_PEOPLE_DONG_202105.csv', 'LOCAL_PEOPLE_DONG_202106.csv',
              'LOCAL_PEOPLE_DONG_202107.csv', 'LOCAL_PEOPLE_DONG_202108.csv',
              'LOCAL_PEOPLE_DONG_202109.csv', 'LOCAL_PEOPLE_DONG_202110.csv']

df_list = []
for f in files_list:
    temp = pd.read_csv(base_path + f, encoding='utf-8', index_col=False)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
print(df.shape)

(1872384, 32)


In [ ]:
print(df['행정동코드'].head())
print(df['행정동코드'].dtype)

0    11215780
1    11110560
2    11260600
3    11200615
4    11590640
Name: 행정동코드, dtype: int64
int64


In [ ]:
elderly_cols = ['남자60세부터64세생활인구수', '남자65세부터69세생활인구수', '남자70세이상생활인구수',
                '여자60세부터64세생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']

df['고령인구수'] = df[elderly_cols].sum(axis=1)

In [ ]:
dong_pop = df.groupby('행정동코드').agg(
    평균총생활인구=('총생활인구수', 'mean'),
    평균고령인구=('고령인구수', 'mean')
).reset_index()

dong_pop['고령인구비율'] = dong_pop['평균고령인구'] / dong_pop['평균총생활인구']

print(dong_pop.shape)
dong_pop.head(10)

(424, 4)


,행정동코드,평균총생활인구,평균고령인구,고령인구비율
0,11110515,16494.444663,3321.372540,0.201363
1,11110530,22580.609569,4200.790873,0.186035
2,11110540,5543.929622,1086.231322,0.195932
3,11110550,14771.669743,3876.777238,0.262447
4,11110560,18990.258605,5381.123898,0.283362
5,11110570,6965.350494,1827.745927,0.262405
6,11110580,7406.807353,1568.681664,0.211789
7,11110600,6654.491685,1200.437789,0.180395
8,11110615,54617.225187,10741.486702,0.196668
9,11110630,16046.023679,4323.711219,0.269457


In [ ]:
print(dong_pop['행정동코드'].astype(str).str.len().unique())

[8]


In [ ]:
print(dong_pop[dong_pop['행정동코드'].astype(str).str.contains('1101')])

Empty DataFrame
Columns: [행정동코드, 평균총생활인구, 평균고령인구, 고령인구비율]
Index: []


In [ ]:
mapping = pd.read_excel('/content/drive/MyDrive/진로탐색프로젝트/민서/행정동코드_매핑정보_20241218.xlsx',
                         sheet_name='행정동코드', skiprows=1)
mapping.columns = ['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명']
mapping.head()

,통계청코드,행자부코드,시도명,시군구명,행정동명
0,1101053,11110530,서울,종로구,사직동
1,1101054,11110540,서울,종로구,삼청동
2,1101055,11110550,서울,종로구,부암동
3,1101056,11110560,서울,종로구,평창동
4,1101057,11110570,서울,종로구,무악동


In [ ]:
dong_pop = dong_pop.merge(
    mapping[['통계청코드', '행자부코드']],
    left_on='행정동코드', right_on='행자부코드', how='left'
)
print(dong_pop.isna().sum())
dong_pop.head()

행정동코드      0
평균총생활인구    0
평균고령인구     0
고령인구비율     0
통계청코드      0
행자부코드      0
dtype: int64


,행정동코드,평균총생활인구,평균고령인구,고령인구비율,통계청코드,행자부코드
0,11110515,16494.444663,3321.372540,0.201363,1101072,11110515
1,11110530,22580.609569,4200.790873,0.186035,1101053,11110530
2,11110540,5543.929622,1086.231322,0.195932,1101054,11110540
3,11110550,14771.669743,3876.777238,0.262447,1101055,11110550
4,11110560,18990.258605,5381.123898,0.283362,1101056,11110560


In [ ]:
dong_summary = result.groupby(['code', 'name']).agg(
    총이용량_GB=('AP별 이용량(GB)', 'sum'),
    AP개수=('관리번호', 'count')
).reset_index()

유형별_AP = result.groupby(['code', '설치유형_대분류']).size().unstack(fill_value=0).reset_index()
유형별_AP.columns.name = None

dong_summary['code'] = dong_summary['code'].astype(int)
유형별_AP['code'] = 유형별_AP['code'].astype(int)
dong_summary = dong_summary.merge(유형별_AP, on='code', how='left')

print(dong_summary.shape)
print(dong_summary.columns.tolist())

(395, 11)
['code', 'name', '총이용량_GB', 'AP개수', '거리', '공공', '공원', '교통', '문화관광', '복지', '상업']


### 행정동코드 불일치 수정

행정동 경계 데이터(2013년 기준)와 행정동코드 매핑정보(행자부 기준)의 코드 체계 차이로 인해
답십리1동(1106085→1106091), 장지동(1124070→1124081) 두 동의 코드를 직접 수정.

In [ ]:
dong_summary.loc[dong_summary['name'] == '답십리1동', 'code'] = 1106091
dong_summary.loc[dong_summary['name'] == '장지동', 'code'] = 1124081

In [ ]:
final = pd.merge(
    dong_summary,
    dong_pop,
    left_on='code', right_on='통계청코드', how='inner'
)
print(final.shape)
final.head(10)

(395, 10)


,code,name,총이용량_GB,AP개수,행정동코드,평균총생활인구,평균고령인구,고령인구비율,통계청코드,행자부코드
0,1101053,사직동,93269.978605,246,11110530,22580.609569,4200.790873,0.186035,1101053,11110530
1,1101054,삼청동,57.089743,1,11110540,5543.929622,1086.231322,0.195932,1101054,11110540
2,1101055,부암동,29085.180390,53,11110550,14771.669743,3876.777238,0.262447,1101055,11110550
3,1101056,평창동,1087.222709,18,11110560,18990.258605,5381.123898,0.283362,1101056,11110560
4,1101057,무악동,3737.344281,10,11110570,6965.350494,1827.745927,0.262405,1101057,11110570
5,1101058,교남동,119.451532,1,11110580,7406.807353,1568.681664,0.211789,1101058,11110580
6,1101060,가회동,1041.933508,9,11110600,6654.491685,1200.437789,0.180395,1101060,11110600
7,1101061,종로1·2·3·4가동,7838.411741,36,11110615,54617.225187,10741.486702,0.196668,1101061,11110615
8,1101063,종로5·6가동,1462.391924,18,11110630,16046.023679,4323.711219,0.269457,1101063,11110630
9,1101064,이화동,1152.575947,10,11110640,22605.810777,4391.687153,0.194272,1101064,11110640


In [ ]:
final.to_csv(base_path + '와이파이_생활인구_통합.csv', index=False, encoding='utf-8-sig')

## 3일차 — 신규 후보지 분리 및 2026년 생활인구 처리

서울시 전체 행정동(424개) 중 기존지점(395개)을 제외한 **29개 동**을 신규 후보지로 분리.
2026년 5월 생활인구 데이터를 동일한 방식으로 처리하여
신규후보지의 현재 시점 생활인구 및 고령인구비율을 산출.

In [ ]:
base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'

기존지점 = pd.read_csv(base_path + '와이파이_생활인구_통합.csv', encoding='utf-8')
print(기존지점.shape)
기존지점.head()

(395, 10)


,code,name,총이용량_GB,AP개수,행정동코드,평균총생활인구,평균고령인구,고령인구비율,통계청코드,행자부코드
0,1101053,사직동,93269.978605,246,11110530,22580.609569,4200.790873,0.186035,1101053,11110530
1,1101054,삼청동,57.089743,1,11110540,5543.929622,1086.231322,0.195932,1101054,11110540
2,1101055,부암동,29085.180390,53,11110550,14771.669743,3876.777238,0.262447,1101055,11110550
3,1101056,평창동,1087.222709,18,11110560,18990.258605,5381.123898,0.283362,1101056,11110560
4,1101057,무악동,3737.344281,10,11110570,6965.350494,1827.745927,0.262405,1101057,11110570


In [ ]:
mapping = pd.read_excel(base_path + '행정동코드_매핑정보_20241218.xlsx',
                         sheet_name='행정동코드', skiprows=1)
mapping.columns = ['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명']
print(mapping.shape)
mapping.head()

(424, 5)


,통계청코드,행자부코드,시도명,시군구명,행정동명
0,1101053,11110530,서울,종로구,사직동
1,1101054,11110540,서울,종로구,삼청동
2,1101055,11110550,서울,종로구,부암동
3,1101056,11110560,서울,종로구,평창동
4,1101057,11110570,서울,종로구,무악동


In [ ]:
전체동 = set(mapping['통계청코드'])
기존동 = set(기존지점['code'])

신규후보지_코드 = 전체동 - 기존동
print(len(신규후보지_코드), "개 신규 후보지")

29 개 신규 후보지


In [ ]:
신규후보지 = mapping[mapping['통계청코드'].isin(신규후보지_코드)]
print(신규후보지.shape)
신규후보지.head(10)

(29, 5)


,통계청코드,행자부코드,시도명,시군구명,행정동명
37,1103059,11170590,서울,용산구,용문동
80,1106071,11230710,서울,동대문구,회기동
81,1106072,11230720,서울,동대문구,휘경1동
82,1106073,11230730,서울,동대문구,휘경2동
86,1106083,11230560,서울,동대문구,전농1동
114,1108062,11290620,서울,성북구,정릉1동
127,1108083,11290575,서울,성북구,동선동
194,1113064,11410640,서울,서대문구,홍제3동
292,1119066,11560660,서울,영등포구,신길4동
295,1119069,11560690,서울,영등포구,신길7동


In [ ]:
df_2026 = pd.read_csv(base_path + 'LOCAL_PEOPLE_DONG_202605.csv', encoding='utf-8', index_col=False)
print(df_2026.shape)
df_2026.head()

(315456, 32)


,기준일ID,시간대구분,행정동코드,총생활인구수,남자0세부터9세생활인구수,남자10세부터14세생활인구수,남자15세부터19세생활인구수,남자20세부터24세생활인구수,남자25세부터29세생활인구수,남자30세부터34세생활인구수,...,여자25세부터29세생활인구수,여자30세부터34세생활인구수,여자35세부터39세생활인구수,여자40세부터44세생활인구수,여자45세부터49세생활인구수,여자50세부터54세생활인구수,여자55세부터59세생활인구수,여자60세부터64세생활인구수,여자65세부터69세생활인구수,여자70세이상생활인구수
0,20260501,0,11440590,24553.5551,1198.5623,537.9692,638.4421,496.0490,588.6774,774.2210,...,812.4782,908.7166,1483.1778,1341.8859,1299.0703,1037.5694,796.3287,602.9879,516.7949,1311.1659
1,20260501,0,11110515,14512.1163,474.3802,302.2761,439.8027,437.2978,497.5920,471.2417,...,383.0333,521.3550,617.9594,730.9672,757.2840,688.9693,574.2482,516.9718,334.1666,1101.8948
2,20260501,0,11110530,11064.9305,219.4512,97.5339,159.7615,253.3513,367.5107,414.5648,...,460.5128,439.9159,494.0747,612.4827,565.5715,495.4564,472.7224,366.6605,231.3208,819.6909
3,20260501,0,11110540,4024.4599,75.0561,57.5107,153.3063,557.2295,167.1259,134.4083,...,85.1815,98.4268,147.1185,152.5695,182.7589,168.0097,105.6623,105.8229,92.5354,302.2245
4,20260501,0,11110550,11158.7235,288.3150,223.5554,354.0519,411.1920,301.9394,232.5133,...,326.7924,326.5727,425.6704,484.7016,549.4437,509.5486,546.3401,409.9723,346.2504,971.1266


In [ ]:
elderly_cols = ['남자60세부터64세생활인구수', '남자65세부터69세생활인구수', '남자70세이상생활인구수',
                '여자60세부터64세생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']

df_2026['고령인구수'] = df_2026[elderly_cols].sum(axis=1)

In [ ]:
dong_pop_2026 = df_2026.groupby('행정동코드').agg(
    평균총생활인구_2026=('총생활인구수', 'mean'),
    평균고령인구_2026=('고령인구수', 'mean')
).reset_index()

dong_pop_2026['고령인구비율_2026'] = dong_pop_2026['평균고령인구_2026'] / dong_pop_2026['평균총생활인구_2026']

print(dong_pop_2026.shape)
dong_pop_2026.head()

(424, 4)


,행정동코드,평균총생활인구_2026,평균고령인구_2026,고령인구비율_2026
0,11110515,17792.927579,3200.638540,0.179883
1,11110530,23617.037011,3983.958069,0.168690
2,11110540,7175.329639,1239.925812,0.172804
3,11110550,11338.524737,2573.665574,0.226984
4,11110560,18571.901135,5213.655939,0.280728


In [ ]:
dong_pop_2026 = dong_pop_2026.merge(
    mapping[['통계청코드', '행자부코드']],
    left_on='행정동코드', right_on='행자부코드', how='left'
)
print(dong_pop_2026.isna().sum())
dong_pop_2026.head()

행정동코드           0
평균총생활인구_2026    0
평균고령인구_2026     0
고령인구비율_2026     0
통계청코드           0
행자부코드           0
dtype: int64


,행정동코드,평균총생활인구_2026,평균고령인구_2026,고령인구비율_2026,통계청코드,행자부코드
0,11110515,17792.927579,3200.638540,0.179883,1101072,11110515
1,11110530,23617.037011,3983.958069,0.168690,1101053,11110530
2,11110540,7175.329639,1239.925812,0.172804,1101054,11110540
3,11110550,11338.524737,2573.665574,0.226984,1101055,11110550
4,11110560,18571.901135,5213.655939,0.280728,1101056,11110560


In [ ]:
신규후보지_최종 = 신규후보지.merge(
    dong_pop_2026[['통계청코드', '평균총생활인구_2026', '평균고령인구_2026', '고령인구비율_2026']],
    on='통계청코드', how='left'
)
print(신규후보지_최종.shape)
신규후보지_최종.head(10)

(29, 8)


,통계청코드,행자부코드,시도명,시군구명,행정동명,평균총생활인구_2026,평균고령인구_2026,고령인구비율_2026
0,1103059,11170590,서울,용산구,용문동,7090.609494,1703.655000,0.240269
1,1106071,11230710,서울,동대문구,회기동,19850.512036,2576.259267,0.129783
2,1106072,11230720,서울,동대문구,휘경1동,20080.928606,3450.266040,0.171818
3,1106073,11230730,서울,동대문구,휘경2동,18471.099683,3567.502672,0.193140
4,1106083,11230560,서울,동대문구,전농1동,36304.223745,7468.070706,0.205708
5,1108062,11290620,서울,성북구,정릉1동,12460.724401,2840.066062,0.227921
6,1108083,11290575,서울,성북구,동선동,24215.666321,3408.814128,0.140769
7,1113064,11410640,서울,서대문구,홍제3동,14311.070506,4146.168514,0.289718
8,1119066,11560660,서울,영등포구,신길4동,6631.284689,1813.932539,0.273542
9,1119069,11560690,서울,영등포구,신길7동,18049.327863,3308.253459,0.183290


In [ ]:
신규후보지_최종.to_csv(base_path + '신규후보지_2026.csv', index=False, encoding='utf-8-sig')

## 4일차 — 예측 모델 구축

입력변수: 평균총생활인구, AP개수, 공공(공공시설 AP), 거리(주요거리 AP) / 타깃: 총이용량_GB
단순회귀를 베이스라인으로, Random Forest, Gradient Boosting,
가우시안 프로세스, GWR 총 5개 모델 비교.
(교수님 피드백에 따라 AP 설치유형 대분류 변수 추가 적용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'
기존지점 = pd.read_csv(base_path + '와이파이_생활인구_통합.csv', encoding='utf-8')

print(기존지점.shape)  # (395, 10) 나와야 정상
기존지점.head()

(395, 10)


,code,name,총이용량_GB,AP개수,행정동코드,평균총생활인구,평균고령인구,고령인구비율,통계청코드,행자부코드
0,1101053,사직동,93269.978605,246,11110530,22580.609569,4200.790873,0.186035,1101053,11110530
1,1101054,삼청동,57.089743,1,11110540,5543.929622,1086.231322,0.195932,1101054,11110540
2,1101055,부암동,29085.180390,53,11110550,14771.669743,3876.777238,0.262447,1101055,11110550
3,1101056,평창동,1087.222709,18,11110560,18990.258605,5381.123898,0.283362,1101056,11110560
4,1101057,무악동,3737.344281,10,11110570,6965.350494,1827.745927,0.262405,1101057,11110570


In [ ]:
# 답십리1동, 장지동 코드 수정
dong_summary.loc[dong_summary['code'] == 1106085, 'code'] = 1106091
dong_summary.loc[dong_summary['code'] == 1124070, 'code'] = 1124081

# 생활인구 데이터 불러오기
dong_pop = pd.read_csv(base_path + '와이파이_생활인구_통합.csv', encoding='utf-8')

# dong_summary(유형별 AP 포함)에 생활인구 붙이기
기존지점 = dong_summary.merge(
    dong_pop[['code', '행정동코드', '평균총생활인구', '평균고령인구', '고령인구비율', '통계청코드', '행자부코드']],
    on='code', how='inner'
)

print(기존지점.shape)
print(기존지점.columns.tolist())

# 저장
기존지점.to_csv(base_path + '와이파이_생활인구_통합.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

(395, 17)
['code', 'name', '총이용량_GB', 'AP개수', '거리', '공공', '공원', '교통', '문화관광', '복지', '상업', '행정동코드', '평균총생활인구', '평균고령인구', '고령인구비율', '통계청코드', '행자부코드']
저장 완료


In [ ]:
X = 기존지점[['평균총생활인구', 'AP개수', '공공', '거리']]
y = 기존지점['총이용량_GB']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(316, 4) (79, 4)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("=== 단순회귀 ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("R²:", r2_score(y_test, lr_pred))

=== 단순회귀 ===
RMSE: 6203.571345633549
MAE: 2699.075718333637
R²: 0.8016448740977404


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("=== Random Forest ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("R²:", r2_score(y_test, rf_pred))

=== Random Forest ===
RMSE: 6095.823404630608
MAE: 2650.221812269366
R²: 0.8084753755134266


### 입력변수 선정: AP개수 추가의 필요성

초기 시도에서 평균총생활인구만 입력변수로 사용했을 때 R²=0.014로 예측력이 낮게 나타났다.
이는 총이용량이 생활인구보다 AP개수에 더 크게 좌우되기 때문으로,
AP개수를 추가하자 R²가 0.787로 크게 향상되었다.

In [ ]:
X = 기존지점[['평균총생활인구', 'AP개수']]
y = 기존지점['총이용량_GB']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("=== 단순회귀 (생활인구+AP개수) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("R²:", r2_score(y_test, lr_pred))

=== 단순회귀 (생활인구+AP개수) ===
RMSE: 6434.43363331722
MAE: 3211.1078169428315
R²: 0.7866068290159909


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("=== Random Forest (생활인구+AP개수) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("R²:", r2_score(y_test, rf_pred))

=== Random Forest (생활인구+AP개수) ===
RMSE: 6759.512565688281
MAE: 2731.67394439643
R²: 0.7645001554172041


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

print("=== Gradient Boosting (생활인구+AP개수) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, gb_pred)))
print("MAE:", mean_absolute_error(y_test, gb_pred))
print("R²:", r2_score(y_test, gb_pred))

=== Gradient Boosting (생활인구+AP개수) ===
RMSE: 6539.303637234945
MAE: 2888.8104003990625
R²: 0.7795942745326104


### 가우시안 프로세스 회귀 (Gaussian Process Regression)

예측값과 함께 예측 불확실성(std)을 동시에 산출하는 모델.
신규 후보지처럼 정답이 없는 지역에 대해 예측 신뢰도를 통계적으로 정량화할 수 있다.
(Rasmussen & Williams, 2006)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel, ConstantKernel as C

kernel = C(1.0) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-8, 1e3)) \
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))

gpr_model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15, random_state=42)
gpr_model.fit(X_scaled, y_scaled)

gpr_pred_scaled, gpr_std_scaled = gpr_model.predict(X_test_scaled, return_std=True)
gpr_pred = scaler_y.inverse_transform(gpr_pred_scaled.reshape(-1, 1)).ravel()
gpr_std = gpr_std_scaled * scaler_y.scale_[0]

print("=== Gaussian Process (sigma_0 조정) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, gpr_pred)))
print("MAE:", mean_absolute_error(y_test, gpr_pred))
print("R²:", r2_score(y_test, gpr_pred))
print("평균 예측 불확실성(std):", gpr_std.mean())

=== Gaussian Process (sigma_0 조정) ===
RMSE: 6163.198763056748
MAE: 2685.7775375011365
R²: 0.8042182467688154
평균 예측 불확실성(std): 8635.128442456555


### 지리가중회귀 (GWR, Geographically Weighted Regression)

"같은 변수라도 동네 위치에 따라 영향력이 다르게 작용한다"는 공간적 이질성을 반영하는 모델.
일반 회귀가 서울 전역에 동일한 계수를 적용하는 반면, GWR은 동마다 다른 계수를 추정함.
공공와이파이 입지라는 공간 문제에 방법론적으로 가장 적합한 모델로 판단하여 최종 채택.

In [ ]:
!pip install mgwr libpysal -q

In [ ]:
dong_boundary_proj = dong_boundary.to_crs(epsg=5179)  # 한국 좌표계로 투영 (거리 계산 정확도를 위해)
dong_boundary_proj['centroid'] = dong_boundary_proj.geometry.centroid
dong_boundary_proj['중심_x'] = dong_boundary_proj['centroid'].x
dong_boundary_proj['중심_y'] = dong_boundary_proj['centroid'].y

좌표정보 = dong_boundary_proj[['code', '중심_x', '중심_y']]
좌표정보['code'] = 좌표정보['code'].astype(int)
print(좌표정보.head())

      code           중심_x          중심_y
0  1125074  968863.838549  1.948680e+06
1  1125073  966806.843870  1.949256e+06
2  1125072  967957.929859  1.950021e+06
3  1125071  969031.136225  1.947788e+06
4  1125070  968406.215505  1.946841e+06


/tmp/ipykernel_3514/2784802479.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  좌표정보['code'] = 좌표정보['code'].astype(int)


In [ ]:
기존지점_gwr = 기존지점.merge(좌표정보, on='code', how='left')
print(기존지점_gwr['중심_x'].isna().sum())  # 결측치 없어야 정상

2


In [ ]:
print(기존지점_gwr[기존지점_gwr['중심_x'].isna()][['code', 'name']])

        code   name
83   1106091  답십리1동
374  1124081    장지동


In [ ]:
보정좌표 = dong_boundary_proj[dong_boundary_proj['name'].isin(['답십리1동', '장지동'])][['name', '중심_x', '중심_y']]
print(보정좌표)

for idx, row in 보정좌표.iterrows():
    기존지점_gwr.loc[기존지점_gwr['name'] == row['name'], '중심_x'] = row['중심_x']
    기존지점_gwr.loc[기존지점_gwr['name'] == row['name'], '중심_y'] = row['중심_y']

print(기존지점_gwr['중심_x'].isna().sum())  # 0이어야 정상

      name           중심_x          중심_y
24     장지동  967886.917667  1.942160e+06
334  답십리1동  960553.888691  1.952247e+06
0


In [ ]:
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

coords = list(zip(기존지점_gwr['중심_x'], 기존지점_gwr['중심_y']))
X_gwr = 기존지점_gwr[['평균총생활인구', 'AP개수', '공공', '거리']].values  # 공공·거리 추가
y_gwr = 기존지점_gwr['총이용량_GB'].values.reshape(-1, 1)

selector = Sel_BW(coords, y_gwr, X_gwr)
bw = selector.search()
print("최적 대역폭:", bw)

gwr_model = GWR(coords, y_gwr, X_gwr, bw)
gwr_results = gwr_model.fit()

print("R² (GWR):", gwr_results.R2)

최적 대역폭: 69.0
R² (GWR): 0.8235856711335291


### GWR과 가우시안 프로세스의 역할 구분

- **GWR**: 기존지점 395개 동의 공간적 이질성 분석에 활용.
  동마다 생활인구·AP유형의 영향력이 다르게 작용함을 확인하는 탐색적 모델.

- **가우시안 프로세스**: 신규후보지 29개에 대한 실제 예측값 및
  예측 불확실성(std) 산출에 활용.
  GWR은 공간 구조 특성상 학습 범위 외 지역 예측이 불안정하여,
  신규후보지 예측은 가우시안 프로세스로 수행함.

### GWR 분석 결과: 강서구의 공간적 특이성

강서구(화곡동, 발산동, 우장산동 등) 일대에서 생활인구 계수가 타 지역 대비 현저히 높게 나타났다.
이는 강서구가 생활인구 변화에 와이파이 이용량이 가장 민감하게 반응하는 지역임을 의미하나,
절대적인 AP당 평균이용량은 양천구·강남구보다 낮은 수준이었다(t검정, p=0.00037).
즉 현재 이용량은 낮지만, 향후 인구 증가 시 이용량이 가장 빠르게 증가할 잠재력을 가진 지역이다.

### GWR 평가 방식에 대한 주의사항

GWR의 R²(0.824)는 전체 395개 동을 대상으로 한 **전역 적합도**이며,
나머지 모델들(단순회귀, RF, GB, 가우시안 프로세스)의 R²가
테스트셋 20% 기준으로 평가된 것과 **직접 비교 시 유의해야 한다**.

GWR을 최종 메인 모델로 채택한 근거는 수치적 우위보다,
"동네마다 생활인구의 영향력이 다르게 작용한다"는
**공간적 이질성을 반영하는 방법론적 적합성**에 있다.

구 단위 검증(공간 블록 교차검증)은 가우시안 프로세스로 대신 수행하였으며,
이 결과에서 확인된 한계(이질적 지역에서 예측력 저하)가
신뢰도 점수 도입의 근거가 되었다.

### Gradient Boosting

여러 개의 약한 예측 모델(결정 트리)을 순차적으로 학습시켜,
이전 모델의 오차를 다음 모델이 보완하는 방식으로 성능을 높이는 앙상블 모델.
Random Forest와 달리 병렬이 아닌 순차 학습 방식이라
과적합에 강하고 복잡한 비선형 패턴을 잘 포착함.
이번 입력변수 확장(공공+거리 추가) 후 R²=0.825로
전체 모델 중 가장 높은 성능을 기록하였다.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

print("=== Gradient Boosting ===")
print("RMSE:", np.sqrt(mean_squared_error(y_test, gb_pred)))
print("MAE:", mean_absolute_error(y_test, gb_pred))
print("R²:", r2_score(y_test, gb_pred))

=== Gradient Boosting ===
RMSE: 5828.482582376815
MAE: 2675.6892104434132
R²: 0.8249061583896109


### 강서구 특이성 원인 분석

자치구별 AP당 평균이용량 비교 및 t검정으로 강서구의 절대적 이용 수준과
생활인구 민감도 간의 관계를 통계적으로 검증.
(입력변수 확장 후에도 강서구의 공간적 특이성은 동일하게 유지됨)

In [ ]:
기존지점_gwr['생활인구_계수'] = gwr_results.params[:, 0]  # 평균총생활인구의 동별 계수
기존지점_gwr['AP개수_계수'] = gwr_results.params[:, 1]

print(기존지점_gwr[['name', '생활인구_계수', 'AP개수_계수']].sort_values('생활인구_계수', ascending=False).head(10))

     name      생활인구_계수   AP개수_계수
251  우장산동  6278.680344 -0.139121
240  화곡6동  6240.987092 -0.140690
234  등촌2동  6166.468910 -0.142900
245  발산1동  6026.744640 -0.127813
247  방화1동  6026.226402 -0.128142
248  방화2동  5965.611541 -0.149802
216   목3동  5945.424038 -0.138315
236  화곡본동  5901.779106 -0.136206
249  방화3동  5869.373288 -0.154359
246   공항동  5813.107733 -0.113837


In [ ]:
강서구동들 = ['발산1동', '화곡3동', '신월5동', '우장산동', '공항동', '화곡6동', '화곡본동', '방화1동', '신월3동', '등촌3동']
print(기존지점_gwr[기존지점_gwr['name'].isin(강서구동들)][['name', 'AP개수', '평균총생활인구', '총이용량_GB']])

     name  AP개수       평균총생활인구       총이용량_GB
220  신월3동    38  13977.281893  23896.084073
222  신월5동     8   9479.736842   1024.535817
235  등촌3동    58  39604.639072   7209.414982
236  화곡본동    49  29503.862454   4873.213177
238  화곡3동    41  19060.831783   5546.762398
240  화곡6동   100  20285.573419  26736.158319
245  발산1동    65  35017.641533  20830.070469
246   공항동    35  34238.810413   4853.664968
247  방화1동    29  32294.213652   5077.281671
251  우장산동    58  34636.199365   9425.621089


In [ ]:
mapping = pd.read_excel(base_path + '행정동코드_매핑정보_20241218.xlsx',
                        sheet_name='행정동코드', skiprows=1)
mapping.columns = ['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명']

기존지점_gwr = 기존지점_gwr.merge(
    mapping[['통계청코드', '시군구명']],
    left_on='code', right_on='통계청코드', how='left'
)

print(기존지점_gwr['시군구명'].isna().sum())  # 0이어야 정상

0


In [ ]:
기존지점_gwr['AP당_평균이용량'] = 기존지점_gwr['총이용량_GB'] / 기존지점_gwr['AP개수']

구별평균 = 기존지점_gwr.groupby('시군구명')['AP당_평균이용량'].mean().sort_values(ascending=False)
print(구별평균.head(10))

시군구명
양천구     388.940336
강남구     352.399973
노원구     348.652682
관악구     342.877853
동작구     333.715174
강북구     303.370499
송파구     303.148228
영등포구    302.756904
용산구     283.233277
동대문구    279.832828
Name: AP당_평균이용량, dtype: float64


In [ ]:
from scipy import stats

강서구 = 기존지점_gwr[기존지점_gwr['시군구명'] == '강서구']['AP당_평균이용량']
나머지 = 기존지점_gwr[기존지점_gwr['시군구명'] != '강서구']['AP당_평균이용량']

t_stat, p_value = stats.ttest_ind(강서구, 나머지, equal_var=False)
print("t-통계량:", t_stat)
print("p-value:", p_value)

t-통계량: -3.882807490062517
p-value: 0.00037232730498157633


### 강서구 분석 결론

강서구는 생활인구 변화에 대한 민감도(GWR 계수)는 가장 높으나,
절대적 이용 수준은 양천구·강남구 등에 미치지 못한다.
향후 인구 증가 시 이용량이 가장 빠르게 늘어날 잠재 지역으로 해석된다.

### 최종 모델 선정

5개 모델 비교 결과(입력변수: 평균총생활인구, AP개수, 공공, 거리),
GWR(R²=0.824)을 메인 예측 모델로, 가우시안 프로세스(R²=0.804)를
보조 신뢰도 산출 모델로 채택.
GWR은 공간적 이질성 반영이라는 방법론적 적합성을,
가우시안 프로세스는 예측 불확실성 정량화 기능을 근거로 선정.

### 최종 입력변수

평균총생활인구, AP개수, 공공(공공시설 AP개수), 거리(주요거리 AP개수)
네 가지로 확정.
고령인구비율 추가 시 R² 변화 없어 제외.
AP 설치유형(공공, 거리) 추가로 기존 대비 모든 모델에서 R² 향상 확인
(단순회귀 0.787→0.802, GB 0.780→0.825, GWR 0.799→0.824).

## 5일차 — 모델 평가

### 공간 블록 교차검증(구 단위 검증) 개요

무작위 분할 검증은 같은 구의 동들이 학습·테스트셋에 함께 포함되어
공간적으로 유사한 데이터를 학습에 활용하는 한계가 있다.
이를 보완하기 위해 특정 구 전체를 학습에서 제외한 뒤 예측하는
구 단위 검증(공간 블록 교차검증)을 수행하였다.
검증 시 입력변수는 최종 확정 변수(평균총생활인구, AP개수, 공공, 거리)를 동일하게 적용.
(Roberts et al., 2017; Meyer & Pebesma, 2021)

In [ ]:
기존지점 = 기존지점.merge(
    mapping[['통계청코드', '시군구명']],
    left_on='code', right_on='통계청코드', how='left'
)
print(기존지점['시군구명'].isna().sum())  # 0이어야 정상

0


In [ ]:
print(기존지점['시군구명'].unique())
print(기존지점['시군구명'].nunique())

['종로구' '중구' '용산구' '성동구' '광진구' '동대문구' '중랑구' '성북구' '강북구' '도봉구' '노원구' '은평구'
 '서대문구' '마포구' '양천구' '강서구' '구로구' '금천구' '영등포구' '동작구' '관악구' '서초구' '강남구' '송파구'
 '강동구']
25


In [ ]:
print(기존지점['시군구명'].value_counts())

시군구명
강남구     21
강서구     20
송파구     20
노원구     19
양천구     18
성북구     18
종로구     17
성동구     17
중랑구     16
서초구     16
관악구     16
마포구     16
은평구     16
영등포구    16
중구      15
용산구     15
광진구     15
구로구     15
강동구     15
동작구     14
도봉구     14
서대문구    13
강북구     13
동대문구    10
금천구     10
Name: count, dtype: int64


### 검증용 구 선정

- **강서구(20개 동)**: GWR에서 생활인구 민감도가 특이하게 높게 나타난 구
- **강남구(21개 동)**: 와이파이 이용량이 절대적으로 높은 번화가 구
- **노원구·구로구**: 상대적으로 평범한 특성을 가진 대조 검증용 구 조합

In [ ]:
검증용구 = ['강서구', '강남구']
검증용_데이터 = 기존지점[기존지점['시군구명'].isin(검증용구)]
학습용_데이터 = 기존지점[~기존지점['시군구명'].isin(검증용구)]

print("학습용:", 학습용_데이터.shape)
print("검증용:", 검증용_데이터.shape)

학습용: (354, 19)
검증용: (41, 19)


In [ ]:
X_train_gu = 학습용_데이터[['평균총생활인구', 'AP개수', '공공', '거리']]
y_train_gu = 학습용_데이터['총이용량_GB']

X_valid_gu = 검증용_데이터[['평균총생활인구', 'AP개수', '공공', '거리']]
y_valid_gu = 검증용_데이터['총이용량_GB']

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_model_gu = LinearRegression()
lr_model_gu.fit(X_train_gu, y_train_gu)
pred_gu = lr_model_gu.predict(X_valid_gu)

print("=== 구 단위 검증 (단순회귀) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_valid_gu, pred_gu)))
print("MAE:", mean_absolute_error(y_valid_gu, pred_gu))
print("R²:", r2_score(y_valid_gu, pred_gu))

=== 구 단위 검증 (단순회귀) ===
RMSE: 6423.42221757017
MAE: 4066.3721348874046
R²: 0.10823986209500824


In [ ]:
X_train_scaled = scaler_X.fit_transform(X_train_gu)
X_valid_scaled = scaler_X.transform(X_valid_gu)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel, ConstantKernel as C

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_gu)
X_valid_scaled = scaler_X.transform(X_valid_gu)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_gu.values.reshape(-1, 1)).ravel()

kernel = C(1.0) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-8, 1e3)) \
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))

gpr_model_gu = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
gpr_model_gu.fit(X_train_scaled, y_train_scaled)

pred_scaled, std_scaled = gpr_model_gu.predict(X_valid_scaled, return_std=True)
pred_gu_gpr = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
std_gu = std_scaled * scaler_y.scale_[0]

print("=== 구 단위 검증 (가우시안 프로세스) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_valid_gu, pred_gu_gpr)))
print("MAE:", mean_absolute_error(y_valid_gu, pred_gu_gpr))
print("R²:", r2_score(y_valid_gu, pred_gu_gpr))
print("평균 불확실성(std):", std_gu.mean())

=== 구 단위 검증 (가우시안 프로세스) ===
RMSE: 6130.902570129453
MAE: 3961.266892142107
R²: 0.1876111664610014
평균 불확실성(std): 8569.703004516718


### 검증 1: 강남구·강서구 제외

**검증 방법**
- 학습 데이터에서 강남구(21개 동), 강서구(20개 동)를 통째로 제외
- 나머지 354개 동으로 모델 재학습 → 제외된 41개 동(강남구+강서구)에 적용해 검증

**결과**

| 모델 | RMSE | MAE | R² |
|---|---|---|---|
| 단순회귀 | 6423.42 | 4066.37 | **0.108** |
| 가우시안 프로세스 | 6130.90 | 3961.27 | **0.188** |

(참고: 기존 변수(생활인구+AP개수)만 사용 시 두 모델 모두 R² 약 -1.07~-1.08)
(참고: 무작위 분할 검증 시 두 모델 모두 R² 약 0.80)

**해석**
- 기존 변수만 사용 시 R²가 음수(평균값보다도 못 맞힘)였으나,
  AP 유형 변수(공공+거리) 추가 후 양수로 개선됨
- 그럼에도 무작위 분할 검증(R²≈0.80) 대비 여전히 낮아,
  강남구·강서구의 입지 특성이 다른 지역과 이질적임은 유지됨
- 단순회귀와 가우시안 프로세스 모두 유사한 패턴 → 모델 문제가 아닌 데이터 특성의 문제

**의미 있는 발견**

가우시안 프로세스의 평균 예측 불확실성(std): 8570
→ 모델이 "이 예측은 자신 없다"는 걸 스스로 표시해준 것
→ 신규 후보지 예측 시 신뢰도 점수가 왜 필요한지 직접 증명하는 결과

**결론**

입력변수 확장으로 이질 지역 예측력이 일부 개선되었으나(R² -1.08→+0.108),
여전히 무작위 분할 검증 대비 낮은 성능을 보여
신규 후보지 예측 시 유사도 기반 신뢰도 점수를 함께 제시하는 방법론적 필요성은 유지됨.

In [ ]:
import numpy as np

# 1단계: 좌표 결측 보정
답십리_좌표 = 기존지점_gwr[기존지점_gwr['name'] == '답십리1동'][['중심_x', '중심_y']].values[0]
장지동_좌표 = 기존지점_gwr[기존지점_gwr['name'] == '장지동'][['중심_x', '중심_y']].values[0]

학습용_gwr.loc[학습용_gwr['name'] == '답십리1동', '중심_x'] = 답십리_좌표[0]
학습용_gwr.loc[학습용_gwr['name'] == '답십리1동', '중심_y'] = 답십리_좌표[1]
학습용_gwr.loc[학습용_gwr['name'] == '장지동', '중심_x'] = 장지동_좌표[0]
학습용_gwr.loc[학습용_gwr['name'] == '장지동', '중심_y'] = 장지동_좌표[1]

print("보정 후 결측:", 학습용_gwr['중심_x'].isna().sum())  # 0이어야 정상

# 2단계: GWR 학습
coords_train = np.array(list(zip(학습용_gwr['중심_x'], 학습용_gwr['중심_y'])))
X_gwr_train = 학습용_gwr[['평균총생활인구', 'AP개수', '공공', '거리']].values
y_gwr_train = 학습용_gwr['총이용량_GB'].values.reshape(-1, 1)

selector_gu = Sel_BW(coords_train, y_gwr_train, X_gwr_train)
bw_gu = selector_gu.search()
print("최적 대역폭:", bw_gu)

gwr_model_gu = GWR(coords_train, y_gwr_train, X_gwr_train, bw_gu)
gwr_results_gu = gwr_model_gu.fit()
print("학습 R² (GWR):", gwr_results_gu.R2)

# 3단계: GWR 검증용 예측
coords_test = np.array(list(zip(검증용_gwr['중심_x'], 검증용_gwr['중심_y'])))
X_gwr_test = 검증용_gwr[['평균총생활인구', 'AP개수', '공공', '거리']].values
y_gwr_test = 검증용_gwr['총이용량_GB'].values

gwr_pred_gu = gwr_model_gu.predict(coords_test, X_gwr_test)
pred_vals = gwr_pred_gu.predictions.flatten()

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
print("\n=== 구 단위 검증 (GWR, 강남구·강서구 제외) ===")
print("RMSE:", round(np.sqrt(mean_squared_error(y_gwr_test, pred_vals)), 3))
print("MAE:", round(mean_absolute_error(y_gwr_test, pred_vals), 3))
print("R²:", round(r2_score(y_gwr_test, pred_vals), 3))

보정 후 결측: 0
최적 대역폭: 60.0
학습 R² (GWR): 0.8276243845490043

=== 구 단위 검증 (GWR, 강남구·강서구 제외) ===
RMSE: 9934.263
MAE: 5450.055
R²: -1.133


### GWR 동일 조건 비교 (강남구·강서구 제외)

교수님 피드백에 따라 GWR도 동일한 공간 블록 교차검증 조건으로 평가하였다.

**결과**

| 모델 | RMSE | MAE | R² |
|---|---|---|---|
| 단순회귀 | 6423.42 | 4066.37 | 0.108 |
| 가우시안 프로세스 | 6130.90 | 3961.27 | **0.188** |
| GWR | 9934.26 | 5450.06 | -1.133 |

**해석**

동일한 검증 조건에서 GWR이 단순회귀·가우시안 프로세스보다 낮은 성능을 보였다.
이는 GWR이 "공간적으로 가까운 동들의 패턴"을 참고하는 특성상,
강남구·강서구처럼 특이 지역을 통째로 제외하면 주변 동들의 데이터도 함께 부재하여
예측이 더욱 불안정해지기 때문으로 해석된다.

이 결과는 GWR 채택 근거가 R² 수치의 우위가 아니라,
"동마다 변수 영향력이 다르게 작용한다"는 **공간적 이질성 반영**이라는
방법론적 적합성에 있음을 재확인시켜준다.
신규 후보지 예측(학습 범위 외 지역)에는 가우시안 프로세스를 활용하는 것이
보다 안정적임을 이 비교를 통해 확인하였다.

In [ ]:
# 1단계: 검증용 구 다시 지정 (이번엔 평범한 구로)
검증용구2 = ['노원구', '구로구']

검증용_데이터2 = 기존지점[기존지점['시군구명'].isin(검증용구2)]
학습용_데이터2 = 기존지점[~기존지점['시군구명'].isin(검증용구2)]

print("학습용:", 학습용_데이터2.shape)
print("검증용:", 검증용_데이터2.shape)

학습용: (361, 19)
검증용: (34, 19)


In [ ]:
# 2단계: 입력/타깃 분리
X_train_gu2 = 학습용_데이터2[['평균총생활인구', 'AP개수', '공공', '거리']]
y_train_gu2 = 학습용_데이터2['총이용량_GB']
X_valid_gu2 = 검증용_데이터2[['평균총생활인구', 'AP개수', '공공', '거리']]
y_valid_gu2 = 검증용_데이터2['총이용량_GB']

In [ ]:
# 3단계: 단순회귀
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_model_gu2 = LinearRegression()
lr_model_gu2.fit(X_train_gu2, y_train_gu2)
pred_gu2 = lr_model_gu2.predict(X_valid_gu2)

print("=== 구 단위 검증 2 (노원구·구로구, 단순회귀) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_valid_gu2, pred_gu2)))
print("MAE:", mean_absolute_error(y_valid_gu2, pred_gu2))
print("R²:", r2_score(y_valid_gu2, pred_gu2))

=== 구 단위 검증 2 (노원구·구로구, 단순회귀) ===
RMSE: 4624.879385552932
MAE: 3429.1842174699727
R²: 0.7577386505590791


In [ ]:
# 4단계: 가우시안 프로세스
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel, ConstantKernel as C

scaler_X2 = StandardScaler()
X_train_scaled2 = scaler_X2.fit_transform(X_train_gu2)
X_valid_scaled2 = scaler_X2.transform(X_valid_gu2)

scaler_y2 = StandardScaler()
y_train_scaled2 = scaler_y2.fit_transform(y_train_gu2.values.reshape(-1, 1)).ravel()

kernel2 = C(1.0) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-8, 1e3)) \
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))

gpr_model_gu2 = GaussianProcessRegressor(kernel=kernel2, n_restarts_optimizer=10, random_state=42)
gpr_model_gu2.fit(X_train_scaled2, y_train_scaled2)

pred_scaled2, std_scaled2 = gpr_model_gu2.predict(X_valid_scaled2, return_std=True)
pred_gu2_gpr = scaler_y2.inverse_transform(pred_scaled2.reshape(-1, 1)).ravel()
std_gu2 = std_scaled2 * scaler_y2.scale_[0]

print("=== 구 단위 검증 2 (노원구·구로구, 가우시안 프로세스) ===")
print("RMSE:", np.sqrt(mean_squared_error(y_valid_gu2, pred_gu2_gpr)))
print("MAE:", mean_absolute_error(y_valid_gu2, pred_gu2_gpr))
print("R²:", r2_score(y_valid_gu2, pred_gu2_gpr))
print("평균 불확실성(std):", std_gu2.mean())

=== 구 단위 검증 2 (노원구·구로구, 가우시안 프로세스) ===
RMSE: 4627.62780876621
MAE: 3424.504704188858
R²: 0.7574506281031081
평균 불확실성(std): 8415.72105801689


### 검증 2: 노원구·구로구 제외

**검증 방법**
- 학습 데이터에서 노원구(19개 동), 구로구(15개 동)를 통째로 제외
- 나머지 361개 동으로 모델 재학습 → 제외된 34개 동(노원구+구로구)에 적용해 검증
- 검증 1(강남구·강서구, 특이 지역)과 대조하기 위한 평범한 지역 조합

**결과**

| 모델 | RMSE | MAE | R² |
|---|---|---|---|
| 단순회귀 | 4624.88 | 3429.18 | **0.758** |
| 가우시안 프로세스 | 4627.63 | 3424.50 | **0.757** |

(참고: 기존 변수(생활인구+AP개수)만 사용 시 두 모델 모두 R² 약 0.53)
(참고: 무작위 분할 검증 시 두 모델 모두 R² 약 0.80)

**해석**
- 기존 변수만 사용 시 R²=0.53이었으나, AP 유형 변수(공공+거리) 추가 후 0.76으로 향상됨
- 무작위 분할 검증(R²≈0.80)에 근접한 수준으로, 평범한 지역에서는 모델이 안정적으로 작동함
- 단순회귀와 가우시안 프로세스가 거의 동일한 성능 → 두 모델 모두 평범한 지역에서는 일관되게 잘 작동함

**의미 있는 발견**

가우시안 프로세스의 평균 예측 불확실성(std): 8416
→ 검증 1(강남구·강서구, std=8570)보다 낮아, 평범한 지역에서 모델이 더 자신 있게 예측함
→ 불확실성 수치가 지역 특성에 따라 다르게 나타남을 확인

**결론**

평범한 지역(노원구·구로구)에서는 새 변수 추가 후 R²가 0.53→0.76으로 크게 향상되어
무작위 분할 검증에 근접한 성능을 보임.
검증 1(특이 지역, R²=0.108~0.188)과 대조하면, 지역 특성에 따른 예측력 차이가 명확히 존재하며
신규 후보지 예측 시 유사도 기반 신뢰도 점수의 필요성이 재확인됨.

In [ ]:
import numpy as np

# 학습용·검증용 데이터에 좌표 붙이기
학습용_gwr2 = 학습용_데이터2.merge(좌표정보, on='code', how='left')
검증용_gwr2 = 검증용_데이터2.merge(좌표정보, on='code', how='left')

print("학습용 좌표 결측:", 학습용_gwr2['중심_x'].isna().sum())
print("검증용 좌표 결측:", 검증용_gwr2['중심_x'].isna().sum())

# 좌표 결측 보정
답십리_좌표 = 기존지점_gwr[기존지점_gwr['name'] == '답십리1동'][['중심_x', '중심_y']].values[0]
장지동_좌표 = 기존지점_gwr[기존지점_gwr['name'] == '장지동'][['중심_x', '중심_y']].values[0]

for df in [학습용_gwr2, 검증용_gwr2]:
    df.loc[df['name'] == '답십리1동', '중심_x'] = 답십리_좌표[0]
    df.loc[df['name'] == '답십리1동', '중심_y'] = 답십리_좌표[1]
    df.loc[df['name'] == '장지동', '중심_x'] = 장지동_좌표[0]
    df.loc[df['name'] == '장지동', '중심_y'] = 장지동_좌표[1]

print("보정 후 학습용 결측:", 학습용_gwr2['중심_x'].isna().sum())

# GWR 학습
coords_train2 = np.array(list(zip(학습용_gwr2['중심_x'], 학습용_gwr2['중심_y'])))
X_gwr_train2 = 학습용_gwr2[['평균총생활인구', 'AP개수', '공공', '거리']].values
y_gwr_train2 = 학습용_gwr2['총이용량_GB'].values.reshape(-1, 1)

selector_gu2 = Sel_BW(coords_train2, y_gwr_train2, X_gwr_train2)
bw_gu2 = selector_gu2.search()
print("최적 대역폭:", bw_gu2)

gwr_model_gu2 = GWR(coords_train2, y_gwr_train2, X_gwr_train2, bw_gu2)
gwr_results_gu2 = gwr_model_gu2.fit()
print("학습 R² (GWR):", gwr_results_gu2.R2)

# GWR 검증용 예측
coords_test2 = np.array(list(zip(검증용_gwr2['중심_x'], 검증용_gwr2['중심_y'])))
X_gwr_test2 = 검증용_gwr2[['평균총생활인구', 'AP개수', '공공', '거리']].values
y_gwr_test2 = 검증용_gwr2['총이용량_GB'].values

gwr_pred_gu2 = gwr_model_gu2.predict(coords_test2, X_gwr_test2)
pred_vals2 = gwr_pred_gu2.predictions.flatten()

print("\n=== 구 단위 검증 (GWR, 노원구·구로구 제외) ===")
print("RMSE:", round(np.sqrt(mean_squared_error(y_gwr_test2, pred_vals2)), 3))
print("MAE:", round(mean_absolute_error(y_gwr_test2, pred_vals2), 3))
print("R²:", round(r2_score(y_gwr_test2, pred_vals2), 3))

학습용 좌표 결측: 2
검증용 좌표 결측: 0
보정 후 학습용 결측: 0
최적 대역폭: 61.0
학습 R² (GWR): 0.8304335019222964

=== 구 단위 검증 (GWR, 노원구·구로구 제외) ===
RMSE: 4712.817
MAE: 3068.003
R²: 0.748


### GWR 공간 블록 교차검증 결과 (동일 조건 비교)
교수님 피드백에 따라 GWR도 동일한 공간 블록 교차검증 조건으로 평가하였다.

**결과**

| 검증 조합 | 모델 | RMSE | MAE | R² |
|---|---|---|---|---|
| 강남구+강서구 (특이 지역) | 단순회귀 | 6423.42 | 4066.37 | 0.108 |
| 강남구+강서구 (특이 지역) | 가우시안 프로세스 | 6130.90 | 3961.27 | **0.188** |
| 강남구+강서구 (특이 지역) | GWR | 9934.26 | 5450.06 | -1.133 |
| 노원구+구로구 (평범한 지역) | 단순회귀 | 4624.88 | 3429.18 | 0.758 |
| 노원구+구로구 (평범한 지역) | 가우시안 프로세스 | 4627.63 | 3424.50 | **0.757** |
| 노원구+구로구 (평범한 지역) | GWR | 4712.82 | 3068.00 | 0.748 |

**해석**

- 평범한 지역(노원구·구로구)에서는 세 모델이 거의 동등한 성능(R² 0.748~0.758)을 보임
- 특이 지역(강남구·강서구)에서는 GWR(-1.133)이 단순회귀(0.108), 가우시안 프로세스(0.188)보다
  크게 낮아, GWR이 공간 구조 특성상 특이 지역에서 더 취약함을 확인
- 이는 GWR이 공간적으로 가까운 동들의 패턴을 참고하는 특성상,
  특이 지역을 통째로 제외하면 예측이 더욱 불안정해지기 때문으로 해석됨

**결론**

동일 조건 비교 결과, GWR의 채택 근거가 R² 수치의 우위가 아니라
공간적 이질성 반영이라는 방법론적 적합성에 있음이 재확인됨.
신규 후보지 예측에는 가우시안 프로세스가 더 안정적임을 확인하였다.

### 신뢰도 점수 산출 (적용가능영역 개념)

신규 후보지는 실제 이용량 정답이 없으므로, 예측값의 신뢰도를 별도로 산출.
기존 학습 지점 395개와의 입지 조건 유사도를 계산하여,
기존 데이터와 유사한 패턴일수록 높은 신뢰도를 부여.

AP개수 처리: 신규후보지는 와이파이가 없어 AP개수가 없으므로,
기존지점의 생활인구-AP개수 관계를 선형회귀로 추정하여 부여.

### 신뢰도 점수 산출 — 방법론적 근거

본 신뢰도 점수는 Meyer & Pebesma(2021)의 **적용가능영역(Area of Applicability)**
개념을 참고하여 설계하였다. 적용가능영역이란 모델이 학습한 데이터의 특성 공간과
예측 대상 데이터가 얼마나 유사한지를 정량화하는 개념으로,
학습 데이터와 유사한 입력값일수록 예측이 신뢰할 만하다는 원리에 기반한다.

본 연구에서는 이를 두 가지 방식으로 구현하였다.
- **거리 기반 신뢰도**: 기존 학습 지점 395개와의 입지 조건 유사도
  (StandardScaler로 표준화된 유클리드 거리)
- **GP 기반 신뢰도**: 가우시안 프로세스가 예측 시 산출하는 불확실성(std)
  (Rasmussen & Williams, 2006)

두 지표를 동일 가중치(50:50)로 결합하였으며,
가중치 민감도 분석(Spearman 상관계수 0.83~0.85)으로 안정성을 검증하였다.

> **[초기 시도]** 전체 평균 AP개수(23.36)를 동일 적용 → 이후 생활인구 비례 추정 방식으로 개선

In [ ]:
신규후보지 = pd.read_csv(base_path + '신규후보지_2026.csv', encoding='utf-8')
print(신규후보지.shape)
print(신규후보지.columns.tolist())

(29, 8)
['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명', '평균총생활인구_2026', '평균고령인구_2026', '고령인구비율_2026']


In [ ]:
평균_AP개수 = 기존지점['AP개수'].mean()
print("기존 지점 평균 AP개수:", 평균_AP개수)

신규후보지['AP개수'] = 평균_AP개수

기존 지점 평균 AP개수: 23.359493670886074


In [ ]:
print(신규후보지[['행정동명', 'AP개수']].head())

   행정동명       AP개수
0   용문동  23.359494
1   회기동  23.359494
2  휘경1동  23.359494
3  휘경2동  23.359494
4  전농1동  23.359494


In [ ]:
신규후보지 = pd.read_csv(base_path + '신규후보지_2026.csv', encoding='utf-8')
print(신규후보지.columns.tolist())
print(신규후보지.shape)

['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명', '평균총생활인구_2026', '평균고령인구_2026', '고령인구비율_2026']
(29, 8)


### AP개수 추정 방식 개선: 생활인구 비례 추정
전체 평균값 대신, 기존지점의 생활인구-AP개수 관계를 선형회귀로 학습하여
신규후보지마다 생활인구 규모에 맞는 AP개수를 개별 추정.

In [ ]:
from sklearn.linear_model import LinearRegression

# AP개수 추정
ap_model = LinearRegression()
ap_model.fit(기존지점[['평균총생활인구']], 기존지점['AP개수'])
신규후보지['AP개수'] = ap_model.predict(
    신규후보지[['평균총생활인구_2026']].rename(columns={'평균총생활인구_2026': '평균총생활인구'})
)
신규후보지['AP개수'] = 신규후보지['AP개수'].clip(lower=1)

# 공공 AP개수 추정
ap_공공_model = LinearRegression()
ap_공공_model.fit(기존지점[['평균총생활인구']], 기존지점['공공'])
신규후보지['공공'] = ap_공공_model.predict(
    신규후보지[['평균총생활인구_2026']].rename(columns={'평균총생활인구_2026': '평균총생활인구'})
)
신규후보지['공공'] = 신규후보지['공공'].clip(lower=0)

# 거리 AP개수 추정
ap_거리_model = LinearRegression()
ap_거리_model.fit(기존지점[['평균총생활인구']], 기존지점['거리'])
신규후보지['거리'] = ap_거리_model.predict(
    신규후보지[['평균총생활인구_2026']].rename(columns={'평균총생활인구_2026': '평균총생활인구'})
)
신규후보지['거리'] = 신규후보지['거리'].clip(lower=0)

display(신규후보지[['행정동명', '평균총생활인구_2026', 'AP개수', '공공', '거리']])

,행정동명,평균총생활인구_2026,AP개수,공공,거리
0,용문동,7090.609494,14.446156,5.913545,2.882111
1,회기동,19850.512036,21.022101,8.397943,3.910436
2,휘경1동,20080.928606,21.140849,8.442806,3.929005
3,휘경2동,18471.099683,20.311207,8.129367,3.799268
4,전농1동,36304.223745,29.501688,11.601540,5.236446
5,정릉1동,12460.724401,17.213699,6.959125,3.314890
6,동선동,24215.666321,23.271728,9.247855,4.262225
7,홍제3동,14311.070506,18.167294,7.319394,3.464010
8,신길4동,6631.284689,14.209439,5.824112,2.845093
9,신길7동,18049.327863,20.093843,8.047246,3.765277


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import numpy as np

scaler = StandardScaler()
기존_표준 = scaler.fit_transform(기존지점[['평균총생활인구', 'AP개수', '공공', '거리']])

신규_입력 = 신규후보지[['평균총생활인구_2026', 'AP개수', '공공', '거리']].rename(
    columns={'평균총생활인구_2026': '평균총생활인구'}
)
신규_표준 = scaler.transform(신규_입력)

거리행렬 = pairwise_distances(신규_표준, 기존_표준)
최소거리 = 거리행렬.min(axis=1)

신규후보지['최근접거리'] = 최소거리
신규후보지['신뢰도점수'] = 100 * (1 - (신규후보지['최근접거리'] - 신규후보지['최근접거리'].min()) /
                                    (신규후보지['최근접거리'].max() - 신규후보지['최근접거리'].min()))

display(신규후보지[['행정동명', '최근접거리', '신뢰도점수']].sort_values('신뢰도점수', ascending=False))

,행정동명,최근접거리,신뢰도점수
11,청림동,0.122413,100.000000
5,정릉1동,0.193304,88.320684
19,거여2동,0.214953,84.753983
20,방이1동,0.222184,83.562666
15,신원동,0.223506,83.344751
27,둔촌1동,0.231604,82.010597
26,천호1동,0.235965,81.292189
24,잠실4동,0.237089,81.107004
7,홍제3동,0.239604,80.692653
0,용문동,0.242914,80.147344


### AP개수 추정 방식 변경에 따른 신뢰도 재산출

초기에는 전체 평균 AP개수(23.36)를 모든 신규후보지에 동일하게 적용했으나,
생활인구에 비례한 회귀 추정 방식으로 변경하여 현실성을 높였다.
또한 교수님 피드백에 따라 추가된 공공·거리 AP개수도 동일하게
생활인구 기반 회귀 추정으로 신규후보지에 부여하였다.
두 방식 모두에서 위례동의 최종신뢰도가 최하위로 유지되어,
이는 단순 가정의 부작용이 아닌 입지 조건의 본질적 이질성에서 비롯된 것으로 해석된다.

In [ ]:
신규_표준_gpr = scaler_X.transform(신규_입력)

pred_scaled_new, std_scaled_new = gpr_model.predict(신규_표준_gpr, return_std=True)

신규후보지['예측이용량'] = scaler_y.inverse_transform(pred_scaled_new.reshape(-1, 1)).ravel()
신규후보지['예측불확실성'] = std_scaled_new * scaler_y.scale_[0]

display(신규후보지[['행정동명', '예측이용량', '예측불확실성']].sort_values('예측불확실성'))

,행정동명,예측이용량,예측불확실성
17,방배3동,6677.128556,8864.226996
6,동선동,6678.117939,8864.228302
16,반포3동,6731.964648,8864.329356
28,둔촌2동,6800.946569,8864.544806
14,남현동,6391.397605,8864.681280
23,가락2동,6389.070863,8864.691782
13,인헌동,6382.115000,8864.723831
12,낙성대동,6849.995634,8864.756753
2,휘경1동,6238.067484,8865.608252
1,회기동,6213.544788,8865.800756


In [ ]:
신규후보지['GP신뢰도'] = 100 * (1 - (신규후보지['예측불확실성'] - 신규후보지['예측불확실성'].min()) /
                                  (신규후보지['예측불확실성'].max() - 신규후보지['예측불확실성'].min()))

신규후보지['최종신뢰도'] = (신규후보지['신뢰도점수'] + 신규후보지['GP신뢰도']) / 2

display(신규후보지[['행정동명', '신뢰도점수', 'GP신뢰도', '최종신뢰도', '예측이용량']].sort_values('최종신뢰도', ascending=False))

,행정동명,신뢰도점수,GP신뢰도,최종신뢰도,예측이용량
27,둔촌1동,82.010597,97.052399,89.531498,7056.644376
24,잠실4동,81.107004,97.576448,89.341726,7016.154856
26,천호1동,81.292189,95.725981,88.509085,7145.559822
1,회기동,77.928217,97.631870,87.780044,6213.544788
2,휘경1동,77.360892,97.921542,87.641217,6238.067484
20,방이1동,83.562666,91.393311,87.477989,5859.008416
19,거여2동,84.753983,90.133776,87.443879,5806.108492
11,청림동,100.000000,74.390258,87.195129,5315.433325
18,개포2동,78.905926,94.727936,86.816931,7203.714850
10,상도4동,77.079771,95.982927,86.531349,6095.646796


In [ ]:
from IPython.display import display

display(신규후보지[신규후보지['행정동명'] == '위례동'])
print()
print("기존지점 평균총생활인구 범위:", 기존지점['평균총생활인구'].min(), "~", 기존지점['평균총생활인구'].max())
print("기존지점 AP개수 범위:", 기존지점['AP개수'].min(), "~", 기존지점['AP개수'].max())

,통계청코드,행자부코드,시도명,시군구명,행정동명,평균총생활인구_2026,평균고령인구_2026,고령인구비율_2026,AP개수,공공,거리,최근접거리,신뢰도점수,예측이용량,예측불확실성,GP신뢰도,최종신뢰도
25,1124082,11710647,서울,송파구,위례동,47695.584661,6753.728962,0.141601,35.372341,13.819479,6.15448,0.729389,0.0,9177.030674,8930.682836,0.0,0.0



기존지점 평균총생활인구 범위: 4801.92852651721 ~ 96798.87533598278
기존지점 AP개수 범위: 1 ~ 329


In [ ]:
from IPython.display import display

display(신규후보지[['행정동명', '고령인구비율_2026']].sort_values('고령인구비율_2026'))
print()
print("기존지점 고령인구비율 범위:", 기존지점['고령인구비율'].min(), "~", 기존지점['고령인구비율'].max())

,행정동명,고령인구비율_2026
12,낙성대동,0.117625
1,회기동,0.129783
6,동선동,0.140769
25,위례동,0.141601
2,휘경1동,0.171818
16,반포3동,0.171927
15,신원동,0.179720
9,신길7동,0.183290
13,인헌동,0.188850
3,휘경2동,0.193140



기존지점 고령인구비율 범위: 0.112087708893935 ~ 0.3602231720357374


> 중간 저장 (신뢰도 점수 산출 완료 시점)

In [ ]:
신규후보지.to_csv(base_path + '신규후보지_최종예측.csv', index=False, encoding='utf-8-sig')

### 위례동 신뢰도 분석 (최종 결과)

신뢰도 점수는 **평균총생활인구, AP개수, 공공, 거리**(생활인구 기반 회귀 추정값)
네 변수의 조합을 보고 계산함

**위례동 데이터**
- 평균총생활인구: 47,695명 → 기존지점 범위(4,802 ~ 96,799) 내에 있음, 단독으로는 극단값 아님
- AP개수(추정): 35.4개, 공공(추정): 13.8개, 거리(추정): 6.2개
  → 생활인구가 가장 많은 만큼 모든 추정값이 신규후보지 중 최대

**핵심 발견: 변수 추가 후에도 결론은 유지됨**

교수님 피드백에 따라 공공·거리 변수를 추가하고 추정 방식도 보완했음에도,
위례동의 거리 기반 신뢰도(0점)와 GP 기반 신뢰도(0점) 모두 최하위로 유지되어
최종신뢰도 0점(전체 최하위)으로 산출됨.
이는 변수 구성이나 가정 방식의 문제가 아니라,
위례동 자체의 입지 조건이 기존 학습 데이터와 본질적으로 이질적이기 때문임을 시사함.

**해석**

위례동은 비교적 최근 조성된 신도시로, 신축 대단지 입주에 따라 인구 규모는
빠르게 커졌지만, 공공 인프라 확산 속도가 이를 따라가지 못한
전형적인 신도시 패턴일 가능성이 높음.
실제로 위례동의 예측이용량(9,177)은 29개 신규후보지 중 가장 높게 산출되었는데,
동시에 신뢰도는 가장 낮아, "예측값과 신뢰도가 반대 방향으로 나타나는" 대표적 사례임.

**결과보고서 반영 문장**

> 위례동은 본 분석에서 예상 이용효율이 가장 높게 추정되었으나,
> 동시에 예측 신뢰도는 가장 낮게 산출된 지역이다.
> 입력변수 확장(공공·거리 추가) 및 추정 방식 개선 후에도
> 이러한 경향이 일관되게 나타나, 이는 분석상의 가정에 따른 우연이 아니라
> 위례동의 입지 조건이 기존 공공와이파이 설치 지점과
> 본질적으로 이질적이기 때문으로 해석된다.
> 비교적 최근 조성된 신도시로서 인구 규모에 비해 공공 인프라 확산이
> 뒤따르지 못한 결과로 추정되며, 이는 "예측값과 신뢰도를 함께 제시해야 한다"는
> 본 연구의 방법론적 필요성을 뒷받침하는 사례다.

### GP 불확실성 정규화 관련 주의사항

신규후보지 29개의 원래 예측불확실성(std)은 8864~8931로 범위가 매우 좁다.
0~100점으로 정규화하면 미세한 차이가 과장되어 표현되는 효과가 있으므로,
GP 기반 신뢰도 점수는 절대적 수치보다 상대적 순위를 참고하는 것이 적절하다.
다만 위례동(8930.68)은 다른 동들(8864~8897)과 차이가 비교적 커서,
GP 기반 신뢰도 0점이 단순 정규화 효과만은 아님을 확인할 수 있다.

In [ ]:
print(scaler_X.mean_)
print(기존지점[['평균총생활인구', 'AP개수', '공공', '거리']].mean())

[2.35547662e+04 2.22711864e+01 9.62994350e+00 3.44632768e+00]
평균총생활인구    24385.966471
AP개수          23.359494
공공             9.281013
거리             4.275949
dtype: float64


In [ ]:
# scaler_X를 전체 기존지점으로 다시 fit
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_전체 = 기존지점[['평균총생활인구', 'AP개수', '공공', '거리']]
X_전체_scaled = scaler_X.fit_transform(X_전체)

scaler_y = StandardScaler()
y_전체_scaled = scaler_y.fit_transform(기존지점['총이용량_GB'].values.reshape(-1, 1)).ravel()

# 확인
print(scaler_X.mean_)
print(기존지점[['평균총생활인구', 'AP개수', '공공', '거리']].mean())

[2.43859665e+04 2.33594937e+01 9.28101266e+00 4.27594937e+00]
평균총생활인구    24385.966471
AP개수          23.359494
공공             9.281013
거리             4.275949
dtype: float64


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel, ConstantKernel as C

kernel = C(1.0) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-8, 1e3)) \
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))

gpr_model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15, random_state=42)
gpr_model.fit(X_전체_scaled, y_전체_scaled)

print("n_features_in_:", gpr_model.n_features_in_)  # 4가 나와야 정상

n_features_in_: 4


In [ ]:
신규_표준_gpr = scaler_X.transform(신규_입력)

pred_scaled_new, std_scaled_new = gpr_model.predict(신규_표준_gpr, return_std=True)

신규후보지['예측이용량'] = scaler_y.inverse_transform(pred_scaled_new.reshape(-1, 1)).ravel()
신규후보지['예측불확실성'] = std_scaled_new * scaler_y.scale_[0]

신규후보지['GP신뢰도'] = 100 * (1 - (신규후보지['예측불확실성'] - 신규후보지['예측불확실성'].min()) /
                                  (신규후보지['예측불확실성'].max() - 신규후보지['예측불확실성'].min()))

신규후보지['최종신뢰도'] = (신규후보지['신뢰도점수'] + 신규후보지['GP신뢰도']) / 2

display(신규후보지[['행정동명', '신뢰도점수', 'GP신뢰도', '최종신뢰도', '예측이용량']].sort_values('최종신뢰도', ascending=False))

,행정동명,신뢰도점수,GP신뢰도,최종신뢰도,예측이용량
27,둔촌1동,82.010597,97.889639,89.950118,6861.635650
24,잠실4동,81.107004,98.338299,89.722651,6821.276432
26,천호1동,81.292189,96.717013,89.004601,6950.264950
18,개포2동,78.905926,95.810802,87.358364,7008.232824
1,회기동,77.928217,96.210319,87.069268,6021.249314
2,휘경1동,77.360892,96.586093,86.973493,6045.693091
20,방이1동,83.562666,88.590301,86.076484,5667.853906
19,거여2동,84.753983,87.102699,85.928341,5615.124224
10,상도4동,77.079771,94.130358,85.605065,5903.730739
12,낙성대동,70.827354,99.620273,85.223813,6655.651942


### 검증 1: scaler_X가 정확한 기준으로 학습되었는지 확인

신뢰도 점수 계산에 쓰인 `scaler_X`가 기존지점(395개 동) 데이터로
정상적으로 학습되었는지 확인

**검증 방법**
- `scaler_X.mean_` (스케일러가 기억하는 평균값)과 기존지점의 실제 평균값을 직접 비교

**결과**
- scaler_X.mean_: [24385.97, 23.36, 9.28, 4.28]
- 기존지점 실제 평균: [24385.97, 23.36, 9.28, 4.28]

**결론**

두 값이 정확히 일치함 → scaler_X는 기존지점 395개 데이터를 기준으로
정상적으로 학습됨. 입력변수 확장(공공·거리 추가) 후에도
전체 기존지점 기준으로 정상 학습되었음을 확인.

In [ ]:
# 검증 2: GP 불확실성 변별력 확인
import numpy as np

평범한_입력 = np.array([[기존지점['평균총생활인구'].median(),
                         기존지점['AP개수'].median(),
                         기존지점['공공'].median(),
                         기존지점['거리'].median()]])
극단적_입력 = np.array([[기존지점['평균총생활인구'].max() * 3,
                         기존지점['AP개수'].max() * 3,
                         기존지점['공공'].max() * 3,
                         기존지점['거리'].max() * 3]])

평범_표준 = scaler_X.transform(평범한_입력)
극단_표준 = scaler_X.transform(극단적_입력)

_, 평범_std = gpr_model.predict(평범_표준, return_std=True)
_, 극단_std = gpr_model.predict(극단_표준, return_std=True)

print("평범한 입력의 불확실성:", 평범_std[0])
print("극단적 입력의 불확실성:", 극단_std[0])

평범한 입력의 불확실성: 0.5558341572922157
극단적 입력의 불확실성: 1.300931829606258


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


### 검증 2: 가우시안 프로세스 불확실성의 변별력 확인

신규후보지 29개의 예측불확실성(std)이 8864~8931로 매우 좁은 범위에 몰려있던 점에 대해,
모델이 정상 작동하는지 확인

**검증 방법**
- 기존지점의 중간값(median) 입력과, 기존지점 최댓값의 3배에 해당하는
  극단적 입력을 각각 넣어 불확실성 비교

**결과**
- 평범한 입력(중간값)의 불확실성: 0.556
- 극단적 입력(최댓값의 3배)의 불확실성: 1.301 (약 2.3배)

**해석**

극단적으로 벗어난 입력에서 불확실성이 커지는 방향은 정상적으로 작동함 → 모델 자체가 고장난 것은 아님.
입력변수 확장(공공·거리 추가) 후 변별력이 기존(약 2배)보다 소폭 향상됨(약 2.3배).
다만 변별력이 여전히 강하지는 않아, GP 기반 신뢰도 점수는
절대적 수치보다 상대적 순위를 참고하는 것이 적절함.

**결과보고서 반영 문장**

> 가우시안 프로세스 모델은 학습 범위를 크게 벗어난 입력
> (생활인구·AP개수·공공·거리 각각 3배)에 대해 불확실성이 약 2.3배 증가하는 것으로 나타나,
> 모델이 입력값의 이질성에 일정 부분 반응함을 확인하였다.
> 입력변수 확장 후 변별력이 소폭 향상되었으나(기존 2배→2.3배),
> 절대적 증가폭이 크지 않아 GP 기반 신뢰도 점수는
> 상대적 순위 참고용으로 활용하는 것이 적절하다.

In [ ]:
신규후보지['최종신뢰도_70_30'] = 0.7 * 신규후보지['신뢰도점수'] + 0.3 * 신규후보지['GP신뢰도']
신규후보지['최종신뢰도_30_70'] = 0.3 * 신규후보지['신뢰도점수'] + 0.7 * 신규후보지['GP신뢰도']

비교 = 신규후보지[['행정동명', '최종신뢰도', '최종신뢰도_70_30', '최종신뢰도_30_70']].sort_values('최종신뢰도', ascending=False)
display(비교)

from scipy.stats import spearmanr
corr1, _ = spearmanr(신규후보지['최종신뢰도'], 신규후보지['최종신뢰도_70_30'])
corr2, _ = spearmanr(신규후보지['최종신뢰도'], 신규후보지['최종신뢰도_30_70'])
print("50:50 vs 70:30 순위 상관관계:", corr1)
print("50:50 vs 30:70 순위 상관관계:", corr2)

,행정동명,최종신뢰도,최종신뢰도_70_30,최종신뢰도_30_70
27,둔촌1동,89.950118,86.774309,93.125926
24,잠실4동,89.722651,86.276392,93.168910
26,천호1동,89.004601,85.919636,92.089566
18,개포2동,87.358364,83.977389,90.739340
1,회기동,87.069268,83.412847,90.725688
2,휘경1동,86.973493,83.128452,90.818533
20,방이1동,86.076484,85.070957,87.082011
19,거여2동,85.928341,85.458598,86.398084
10,상도4동,85.605065,82.194947,89.015182
12,낙성대동,85.223813,79.465229,90.982397


50:50 vs 70:30 순위 상관관계: 0.8280788177339901
50:50 vs 30:70 순위 상관관계: 0.8522167487684728


### 검증 3: 거리 기반·GP 기반 신뢰도 가중치(50:50)의 안정성 확인

최종신뢰도를 거리 기반 신뢰도와 GP 기반 신뢰도의 단순 평균(50:50)으로
계산한 것이 임의적인 선택은 아닌지 검증

**검증 방법**
- 가중치를 70:30, 30:70으로 각각 바꿔서 최종신뢰도를 다시 계산
- 원래(50:50) 순위와 Spearman 순위상관계수로 비교

**결과**
- 50:50 vs 70:30 순위 상관관계: 0.828
- 50:50 vs 30:70 순위 상관관계: 0.852

**해석**

가중치를 크게 바꿔도(70:30, 30:70) 전체 순위는 대체로 유지됨(상관계수 0.83 ~ 0.85)
→ 50:50 가중치 선택이 결과에 결정적인 영향을 주는 자의적 선택이 아님을 확인.
이전 입력변수 2개 기준(0.94~0.97)보다 상관계수가 낮아진 것은,
새 변수(공공·거리) 추가로 두 신뢰도 지표 간 상호작용이 복잡해졌기 때문으로 해석됨.

**예외 사례: 위례동**

위례동은 어떤 가중치 조합에서도 0점으로 유지되어,
두 지표 모두 일관되게 최하위로 판단함.

**결과보고서 반영 문장**

> 가중치 민감도 분석을 통해 50:50 가중치 선택의 안정성을 확인하였다
> (50:50 대비 70:30, 30:70 가중치 적용 시 Spearman 순위상관계수 각각 0.828, 0.852).
> 입력변수 확장(공공·거리 추가) 후 상관계수가 이전(0.94~0.97) 대비
> 다소 낮아졌으나, 0.8 이상으로 여전히 안정적인 수준임을 확인하였다.
> 위례동은 두 신뢰도 지표 모두 0점으로 일관되게 최하위로 나타나,
> 거리 기반 방법과 모델 기반 방법이 동일하게 이 지역의 이질성을
> 포착하고 있음을 시사한다.

> 최종 저장 (검증 1~3 완료 후)

In [ ]:
신규후보지.to_csv(base_path + '신규후보지_최종예측.csv', index=False, encoding='utf-8-sig')

### 군집 라벨 의미 부여 (기존지점 395개)

K-means 군집 번호(0,1,2, 3, 4)에 고령인구비율 기준으로 의미 있는 이름을 매핑.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'

df = pd.read_csv(base_path + '행정동_군집분석2.csv', encoding='utf-8')
print(df.shape)
df.head()

(395, 19)


,code,name,총이용량_GB,AP개수,거리,공공,공원,교통,문화관광,복지,상업,행정동코드,평균총생활인구,평균고령인구,고령인구비율,통계청코드,행자부코드,고령인구_군집,군집명
0,1101053,사직동,93269.978605,246,16,190,2,0,38,0,0,11110530,22580.609569,4200.790873,0.186035,1101053,11110530,1,중저령(양호)
1,1101054,삼청동,57.089743,1,0,0,0,0,1,0,0,11110540,5543.929622,1086.231322,0.195932,1101054,11110540,1,중저령(양호)
2,1101055,부암동,29085.180390,53,2,48,3,0,0,0,0,11110550,14771.669743,3876.777238,0.262447,1101055,11110550,3,중고령(취약)
3,1101056,평창동,1087.222709,18,12,0,3,0,0,3,0,11110560,18990.258605,5381.123898,0.283362,1101056,11110560,0,초고령(매우 취약)
4,1101057,무악동,3737.344281,10,0,0,0,0,10,0,0,11110570,6965.350494,1827.745927,0.262405,1101057,11110570,3,중고령(취약)


### 신규 후보지 군집 라벨 부여

기존지점 학습에 사용한 K-means 모델(k=5)을 그대로 적용하여
신규후보지의 고령인구비율_2026으로 군집 예측(kmeans.predict()).
군집 번호는 연우 노트북 기준과 동일하게 재매핑하여 일관성 유지.

In [7]:
신규후보지 = pd.read_csv(base_path + '신규후보지_최종예측.csv', encoding='utf-8')
print(신규후보지.shape)
print(신규후보지.columns.tolist())

기존지점 = pd.read_csv(base_path + '행정동_군집분석2.csv', encoding='utf-8')
print(기존지점.shape)
print(기존지점.columns.tolist())

(29, 19)
['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명', '평균총생활인구_2026', '평균고령인구_2026', '고령인구비율_2026', 'AP개수', '공공', '거리', '최근접거리', '신뢰도점수', '예측이용량', '예측불확실성', 'GP신뢰도', '최종신뢰도', '최종신뢰도_70_30', '최종신뢰도_30_70']
(395, 19)
['code', 'name', '총이용량_GB', 'AP개수', '거리', '공공', '공원', '교통', '문화관광', '복지', '상업', '행정동코드', '평균총생활인구', '평균고령인구', '고령인구비율', '통계청코드', '행자부코드', '고령인구_군집', '군집명']


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 기존지점으로 K-means 재학습 (k=5)
X = 기존지점[['고령인구비율']]

scaler_km = StandardScaler()
X_scaled = scaler_km.fit_transform(X)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# 군집 중심값 확인
print("군집별 고령인구비율 평균:")
기존지점['고령인구_군집_temp'] = kmeans.labels_
print(기존지점.groupby('고령인구_군집_temp')['고령인구비율'].mean().sort_values())

군집별 고령인구비율 평균:
고령인구_군집_temp
4    0.146766
1    0.188658
2    0.227931
3    0.259933
0    0.299830
Name: 고령인구비율, dtype: float64


In [9]:
# 신규후보지에 군집 적용
X_신규 = 신규후보지[['고령인구비율_2026']].rename(columns={'고령인구비율_2026': '고령인구비율'})
X_신규_scaled = scaler_km.transform(X_신규)

신규후보지['고령인구_군집'] = kmeans.predict(X_신규_scaled)

# 군집명 매핑 (연우 파일과 동일하게)
군집명_매핑 = {4: '저령(매우 양호)', 1: '중저령(양호)', 2: '중령(보통)', 3: '중고령(취약)', 0: '초고령(매우 취약)'}
신규후보지['군집명'] = 신규후보지['고령인구_군집'].map(군집명_매핑)

display(신규후보지[['행정동명', '고령인구비율_2026', '고령인구_군집', '군집명']].sort_values('고령인구_군집'))

,행정동명,고령인구비율_2026,고령인구_군집,군집명
7,홍제3동,0.289718,0,초고령(매우 취약)
3,휘경2동,0.193140,1,중저령(양호)
4,전농1동,0.205708,1,중저령(양호)
2,휘경1동,0.171818,1,중저령(양호)
13,인헌동,0.188850,1,중저령(양호)
15,신원동,0.179720,1,중저령(양호)
9,신길7동,0.183290,1,중저령(양호)
16,반포3동,0.171927,1,중저령(양호)
20,방이1동,0.197105,1,중저령(양호)
18,개포2동,0.193504,1,중저령(양호)


In [10]:
신규후보지.to_csv(base_path + '신규후보지_최종예측.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

저장 완료


## 6일차 — 종합 인덱스 산출

K-means 군집분석 결과(기존 395개 동 기준 학습)를 활용하여
신규후보지 29개에도 동일 모델로 군집을 예측·부여.
이후 디지털취약지수, 예상 이용효율, 예측 신뢰도를 결합하여 종합 추천 인덱스 산출.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

base_path = '/content/drive/MyDrive/진로탐색프로젝트/민서/'

# 신규후보지 (예측이용량 + 신뢰도 + 군집 다 들어있는 최종본)
신규후보지 = pd.read_csv(base_path + '신규후보지_최종예측.csv', encoding='utf-8')
print(신규후보지.shape)
print(신규후보지.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(29, 19)
['통계청코드', '행자부코드', '시도명', '시군구명', '행정동명', '평균총생활인구_2026', '평균고령인구_2026', '고령인구비율_2026', 'AP개수', '최근접거리', '신뢰도점수', '예측이용량', '예측불확실성', 'GP신뢰도', '최종신뢰도', '최종신뢰도_70_30', '최종신뢰도_30_70', '고령인구_군집', '군집명']


In [11]:
# 디지털취약지수 점수 매핑 (k=5 기준)
취약지수_매핑 = {
    '초고령(매우 취약)': 100,
    '중고령(취약)': 75,
    '중령(보통)': 50,
    '중저령(양호)': 25,
    '저령(매우 양호)': 0
}
신규후보지['디지털취약지수'] = 신규후보지['군집명'].map(취약지수_매핑)

# 예상이용효율 0~100 정규화
신규후보지['이용효율점수'] = 100 * (신규후보지['예측이용량'] - 신규후보지['예측이용량'].min()) / \
                              (신규후보지['예측이용량'].max() - 신규후보지['예측이용량'].min())

display(신규후보지[['행정동명', '군집명', '디지털취약지수', '이용효율점수', '최종신뢰도']])

,행정동명,군집명,디지털취약지수,이용효율점수,최종신뢰도
0,용문동,중령(보통),50,1.118550,62.517692
1,회기동,저령(매우 양호),0,32.191532,87.069268
2,휘경1동,중저령(양호),25,32.752644,86.973493
3,휘경2동,중저령(양호),25,28.832380,84.772982
4,전농1동,중저령(양호),25,72.259698,63.462315
5,정릉1동,중령(보통),50,14.195882,81.051889
6,동선동,저령(매우 양호),0,42.821579,77.467632
7,홍제3동,초고령(매우 취약),100,18.701855,80.989204
8,신길4동,중고령(취약),75,0.000000,58.267237
9,신길7동,중저령(양호),25,27.805279,83.154099


In [12]:
신규후보지['종합추천인덱스'] = (
    신규후보지['디지털취약지수'] * (1/3) +
    신규후보지['이용효율점수'] * (1/3) +
    신규후보지['최종신뢰도'] * (1/3)
)

display(신규후보지[['행정동명', '시군구명', '군집명', '디지털취약지수', '이용효율점수', '최종신뢰도', '종합추천인덱스']].sort_values('종합추천인덱스', ascending=False))

,행정동명,시군구명,군집명,디지털취약지수,이용효율점수,최종신뢰도,종합추천인덱스
26,천호1동,강동구,중고령(취약),75,53.517266,89.004601,72.507289
28,둔촌2동,강동구,중고령(취약),75,45.632060,83.099006,67.910355
7,홍제3동,서대문구,초고령(매우 취약),100,18.701855,80.989204,66.563686
27,둔촌1동,강동구,중령(보통),50,51.482763,89.950118,63.810960
19,거여2동,송파구,중고령(취약),75,22.868852,85.928341,61.265731
14,남현동,관악구,중령(보통),50,36.261039,85.020644,57.093895
23,가락2동,송파구,중령(보통),50,36.207800,85.025298,57.077700
18,개포2동,강남구,중저령(양호),25,54.847930,87.358364,55.735431
24,잠실4동,송파구,중저령(양호),25,50.556310,89.722651,55.092987
10,상도4동,동작구,중령(보통),50,29.493871,85.605065,55.032978


In [13]:
신규후보지 = 신규후보지.sort_values('종합추천인덱스', ascending=False).reset_index(drop=True)
신규후보지.to_csv(base_path + '신규후보지_최종예측.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

저장 완료


## 6일차 — 종합 추천 인덱스 산출 결과

### 종합 추천 인덱스란?

"신규 후보지 29개 동 중 어디에 공공와이파이를 새로 설치하는 게 가장 효과적인가"를
숫자 하나로 표현한 우선순위 점수.

| 구성 요소 | 사용 변수 | 배점 | 의미 |
|---|---|---|---|
| 디지털취약지수 | 군집명(고령인구비율 기반, k=5) | 1/3 | 이 동네가 디지털적으로 얼마나 취약한가 |
| 예상 이용효율 | 예측이용량(GWR+GP 기반) | 1/3 | 설치하면 얼마나 많이 쓸 것으로 예상되는가 |
| 예측 신뢰도 | 최종신뢰도(거리+GP 기반) | 1/3 | 이 예측을 얼마나 믿을 수 있는가 |

**종합추천인덱스 = (디지털취약지수 + 이용효율점수 + 최종신뢰도) / 3**

세 지표 모두 0~100점으로 정규화한 뒤 동일 가중치(1/3)로 평균냈다.

---

### 디지털취약지수 산출 방법

신연우의 K-means 군집분석 결과를 활용했다.
교수님 피드백에 따라 기존 k=3에서 **k=5로 확장**하여 군집별 점수 차이를 줄였다.
기존 395개 동의 고령인구비율(2021년 5~10월 평균)로 군집을 학습한 뒤,
신규후보지 29개에도 동일 모델(kmeans.predict())로 군집을 부여했다.

| 군집 | 고령인구비율 평균 | 점수 |
|---|---|---|
| 초고령(매우 취약) | 약 30.0% | 100점 |
| 중고령(취약) | 약 26.0% | 75점 |
| 중령(보통) | 약 22.8% | 50점 |
| 중저령(양호) | 약 18.9% | 25점 |
| 저령(매우 양호) | 약 14.7% | 0점 |

기존 k=3(0·50·100점, 50점 차이)에서 k=5(0·25·50·75·100점, 25점 차이)로
점수 간격이 줄어들어 지표 간 균형이 개선되었다.

---

### 예상 이용효율 산출 방법

4~5일차에 구축한 예측모델(GWR 메인, 가우시안 프로세스 보조)로
신규후보지 29개의 예측이용량을 산출했다.
입력변수는 교수님 피드백에 따라 기존(생활인구, AP개수)에서
**공공·거리 AP개수를 추가(생활인구, AP개수, 공공, 거리)**하여 확장하였다.
신규후보지의 AP개수·공공·거리는 기존지점의 생활인구 기반 선형회귀로 추정했다.
이후 예측이용량을 0~100점으로 정규화했다.

> **주의사항**: AP개수 추정 방식은 생활인구 규모에 비례한다는 가정에 기반하며,
> 실제 설치 규모와 차이가 있을 수 있다.

---

### 예측 신뢰도 산출 방법

두 가지 방법을 결합해 최종신뢰도를 산출했다.

- **거리 기반 신뢰도**: 각 신규후보지와 기존 학습 지점 395개 사이의
  입지 조건 유사도(StandardScaler로 표준화된 유클리드 거리, 4개 변수 기준)
- **GP 기반 신뢰도**: 가우시안 프로세스 모델이 예측 시 자체적으로 산출하는
  불확실성(std). 불확실성이 낮을수록 높은 점수.

**최종신뢰도 = (거리 기반 신뢰도 + GP 기반 신뢰도) / 2** (동일 가중치 적용)

가중치 민감도 분석(70:30, 30:70) 결과 Spearman 순위상관계수 0.83~0.85로
50:50 선택의 안정성을 확인했다.

---

### 종합 추천 인덱스 최종 결과

| 순위 | 행정동 | 자치구 | 군집명 | 디지털취약지수 | 이용효율점수 | 최종신뢰도 | 종합추천인덱스 |
|---|---|---|---|---|---|---|---|
| 1 | 천호1동 | 강동구 | 중고령(취약) | 75 | 53.5 | 89.0 | 72.5 |
| 2 | 둔촌2동 | 강동구 | 중고령(취약) | 75 | 45.6 | 83.1 | 67.9 |
| 3 | 홍제3동 | 서대문구 | 초고령(매우 취약) | 100 | 18.7 | 81.0 | 66.6 |
| 4 | 둔촌1동 | 강동구 | 중령(보통) | 50 | 51.5 | 90.0 | 63.8 |
| 5 | 거여2동 | 송파구 | 중고령(취약) | 75 | 22.9 | 85.9 | 61.3 |
| 6 | 남현동 | 관악구 | 중령(보통) | 50 | 36.3 | 85.0 | 57.1 |
| 7 | 가락2동 | 송파구 | 중령(보통) | 50 | 36.2 | 85.0 | 57.1 |
| 8 | 개포2동 | 강남구 | 중저령(양호) | 25 | 54.8 | 87.4 | 55.7 |
| 9 | 잠실4동 | 송파구 | 중저령(양호) | 25 | 50.6 | 89.7 | 55.1 |
| 10 | 상도4동 | 동작구 | 중령(보통) | 50 | 29.5 | 85.6 | 55.0 |
| … | … | … | … | … | … | … | … |
| 29 | 위례동 | 송파구 | 저령(매우 양호) | 0 | 100.0 | 0.0 | 33.3 |

---

### 결과 해석

#### 1) 최우선 설치 추천 지역 — 천호1동·둔촌2동·홍제3동

상위 1~3위는 모두 중고령(취약) 또는 초고령(매우 취약) 군집에 해당하는 동이다.

**천호1동(72.5점)**은 중고령(취약) 군집으로 세 지표가 가장 균형 잡힌 동이다.
이용효율점수(53.5), 신뢰도(89.0), 디지털취약지수(75) 모두 고르게 높아
설치 효과가 크면서 예측도 안정적인 최우선 추천 지역이다.

**둔촌2동(67.9점)**은 중고령(취약) 군집으로 신뢰도(83.1)와 이용효율(45.6)이
균형 잡혀 있어 2위에 올랐다.

**홍제3동(66.6점)**은 초고령(매우 취약) 군집으로 디지털취약지수가 100점이나,
이용효율점수가 낮아 3위에 그쳤다.

#### 2) k=5 확장의 효과

k=3에서는 천호1동이 76.7점으로 1위였으나, k=5로 바꾸면서 72.5점으로 조정됐다.
군집별 점수 차이가 줄어들면서(50점→25점 간격) 이용효율점수와 신뢰도의
상대적 영향력이 커졌고, 결과적으로 순위가 더 다양하게 분포됐다.

#### 3) 신뢰도가 예측값의 영향을 상쇄한 대표 사례 — 위례동

**위례동(33.3점)**은 이용효율점수가 100점으로 신규후보지 중 가장 높지만,
최종신뢰도가 0점으로 종합 순위 최하위를 기록했다.
이는 "예측값이 높다고 무조건 설치 우선순위가 높은 것은 아니다.
그 예측이 얼마나 믿을 만한지를 함께 고려해야 한다"는 본 연구의
핵심 메시지를 잘 보여주는 사례다.

---

### 종합 결론

신규 공공와이파이 설치 우선순위 상위 지역으로
**천호1동, 둔촌2동, 홍제3동(중고령·초고령 취약 지역)** 및
**둔촌1동, 거여2동, 남현동(중령·중고령, 이용효율 및 신뢰도 균형)**을 도출하였다.

본 연구는 교수님 피드백을 반영하여 다음과 같이 보완하였다.
- 입력변수 확장(생활인구+AP개수 → 생활인구+AP개수+공공+거리)으로
  모든 모델 R² 향상(최대 +0.045)
- K-means 군집 수 확장(k=3→k=5)으로 디지털취약지수 점수 간격 조정(50→25점)
- GWR 동일 조건 공간 블록 교차검증 추가

다만 세 지표의 가중치를 동일하게(1/3) 설정한 것은 방법론적 선택이며,
정책 목표에 따라 디지털취약지수에 더 큰 가중치를 부여하거나
이용효율에 더 큰 가중치를 부여하는 등 다양한 변형이 가능하다.

> **방법론적 근거**: 디지털취약지수는 허선영 외(2018)의 취약계층 분류 방식을,
> 공간 블록 교차검증은 Roberts et al.(2017)을,
> 가우시안 프로세스 불확실성 정량화는 Rasmussen & Williams(2006)을,
> 신뢰도 점수는 Meyer & Pebesma(2021)의 적용가능영역 개념을 참고하였다.
> 각 수식적 정의는 실험적 결정이 아닌 관련 문헌에 근거하여 설계되었다.

In [14]:
# 종합 인덱스 가중치 민감도 분석
from scipy.stats import spearmanr

# 가중치 조합별로 종합 인덱스 다시 계산
신규후보지['종합인덱스_취약중심'] = (
    신규후보지['디지털취약지수'] * 0.5 +
    신규후보지['이용효율점수'] * 0.25 +
    신규후보지['최종신뢰도'] * 0.25
)

신규후보지['종합인덱스_효율중심'] = (
    신규후보지['디지털취약지수'] * 0.25 +
    신규후보지['이용효율점수'] * 0.5 +
    신규후보지['최종신뢰도'] * 0.25
)

신규후보지['종합인덱스_신뢰중심'] = (
    신규후보지['디지털취약지수'] * 0.25 +
    신규후보지['이용효율점수'] * 0.25 +
    신규후보지['최종신뢰도'] * 0.5
)

# 원래(1/3씩)와 순위 상관관계 비교
corr1, _ = spearmanr(신규후보지['종합추천인덱스'], 신규후보지['종합인덱스_취약중심'])
corr2, _ = spearmanr(신규후보지['종합추천인덱스'], 신규후보지['종합인덱스_효율중심'])
corr3, _ = spearmanr(신규후보지['종합추천인덱스'], 신규후보지['종합인덱스_신뢰중심'])

print("1/3씩 vs 취약중심(0.5:0.25:0.25) 순위상관계수:", round(corr1, 3))
print("1/3씩 vs 효율중심(0.25:0.5:0.25) 순위상관계수:", round(corr2, 3))
print("1/3씩 vs 신뢰중심(0.25:0.25:0.5) 순위상관계수:", round(corr3, 3))

# 상위 10개 순위 비교
비교표 = 신규후보지[['행정동명', '종합추천인덱스', '종합인덱스_취약중심', '종합인덱스_효율중심', '종합인덱스_신뢰중심']].sort_values('종합추천인덱스', ascending=False).head(10)
display(비교표)

1/3씩 vs 취약중심(0.5:0.25:0.25) 순위상관계수: 0.901
1/3씩 vs 효율중심(0.25:0.5:0.25) 순위상관계수: 0.822
1/3씩 vs 신뢰중심(0.25:0.25:0.5) 순위상관계수: 0.951


,행정동명,종합추천인덱스,종합인덱스_취약중심,종합인덱스_효율중심,종합인덱스_신뢰중심
0,천호1동,72.507289,73.130467,67.759783,76.631617
1,둔촌2동,67.910355,69.682767,62.340782,71.707518
2,홍제3동,66.563686,74.922765,54.598228,70.170066
3,둔촌1동,63.810960,60.358220,60.728911,70.345750
4,거여2동,61.265731,64.699298,51.666511,67.431383
5,남현동,57.093895,55.320421,51.885681,64.075582
6,가락2동,57.077700,55.308275,51.860225,64.064599
7,개포2동,55.735431,48.051574,55.513556,63.641165
8,잠실4동,55.092987,47.569740,53.958818,63.750403
9,상도4동,55.032978,53.774734,48.648201,62.676000


In [15]:
신규후보지.to_csv(base_path + '신규후보지_최종예측.csv', index=False, encoding='utf-8-sig')
print("저장 완료")

저장 완료


### 검증 4: 종합 추천 인덱스 가중치(1/3씩)의 안정성 확인

**검증 방법**
- 세 지표의 가중치를 달리해서(취약중심, 효율중심, 신뢰중심) 종합 인덱스를 다시 계산
- 원래(1/3씩) 순위와 Spearman 순위상관계수로 비교

**결과**

| 가중치 조합 | 순위 상관계수 |
|---|---|
| 1/3씩 vs 취약중심 (0.5:0.25:0.25) | 0.901 |
| 1/3씩 vs 효율중심 (0.25:0.5:0.25) | 0.822 |
| 1/3씩 vs 신뢰중심 (0.25:0.25:0.5) | 0.951 |

**해석**

세 가지 가중치 조합 모두 Spearman 순위상관계수 0.82 이상으로,
가중치를 바꿔도 전체적인 순위는 크게 유지됨.
1/3 동일 가중치 선택이 특정 지표에 편향되지 않은 안정적인 선택임을 확인.

효율중심(0.822)에서 상관계수가 가장 낮아,
이용효율 점수가 순위에 가장 민감하게 작용하는 변수임을 알 수 있음.
천호1동·홍제3동 등 상위권은 어떤 가중치 조합에서도 안정적으로 유지됨.

## 참고문헌

1. 허선영, 김주영, 문태헌. (2018). 머신러닝기반 범죄발생 위험지역 예측.
   한국지리정보학회지, 21(4), 64-80.
   → 디지털취약지수(군집분석) 설계 참고

2. Roberts, D. R. et al. (2017). Cross-validation strategies for data with
   temporal, spatial, hierarchical, or phylogenetic structure.
   Ecography, 40(8), 913-929.
   → 구 단위 공간 블록 교차검증 설계 근거

3. Meyer, H., & Pebesma, E. (2021). Predicting into unknown space?
   Methods in Ecology and Evolution, 12, 1620-1633.
   → 신뢰도 점수(적용가능영역) 설계 근거

4. Rasmussen, C. E., & Williams, C. K. I. (2006).
   Gaussian Processes for Machine Learning. MIT Press.
   → 가우시안 프로세스 모델 채택 및 불확실성 정량화 근거